# 03 Core Experiment

Test how much multi-token structural information is visible on the activation side, and how consistently the same distinction can be recovered through ordinary LM continuation once the first token is supplied.

Pair 1 is used for pilot development. Pairs 2-15 are held out.

In [1]:
import os
import sys
import pandas as pd

REPO_PATH = "/workspace/multi-token-jlens/j-lens"
DATA_PATH = "/workspace/multi-token-jlens/data/final_minimal_pairs.csv"
LENS_PATH = "/workspace/multi-token-jlens/j-lens/lens.pt"

if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

import jlens

dataset = pd.read_csv(DATA_PATH)

print("J-Lens repo imported:", True)
print("Dataset exists:", os.path.exists(DATA_PATH))
print("Lens exists:", os.path.exists(LENS_PATH))
print("Number of pairs:", len(dataset))
print("Number of categories:", dataset["category"].nunique())
print()
print(dataset["category"].value_counts())

J-Lens repo imported: True
Dataset exists: True
Lens exists: True
Number of pairs: 15
Number of categories: 5

category
role_argument_binding    3
modifier_attachment      3
spatial_relation         3
comparative_relation     3
quantifier_binding       3
Name: count, dtype: int64


## Load model and fitted lens

Load the same Qwen3.5-4B model and fitted J-Lens used in the reproduction check.

In [2]:
MODEL = "Qwen/Qwen3.5-4B"

model, tok = jlens.load_model(MODEL)
lens = jlens.JLens.load(LENS_PATH)

print("Model loaded:", type(model).__name__)
print("Tokenizer loaded:", type(tok).__name__)
print("Layers:", len(model.model.layers))
print("Lens loaded:", type(lens).__name__)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Model loaded: Qwen3_5ForCausalLM
Tokenizer loaded: Qwen2Tokenizer
Layers: 32
Lens loaded: JLens


## Canonical structural targets

For each minimal pair, define two short candidate interpretations that represent the relational structure we want to track.

For Pair 1:

`user tests model`  
`model tests user`

In [3]:
pair1_targets = {
    "A": {
        "input": "The user tests the model.",
        "target": "user tests model",
    },
    "B": {
        "input": "The model tests the user.",
        "target": "model tests user",
    },
}

for label, item in pair1_targets.items():
    target_ids = tok.encode(
        item["target"],
        add_special_tokens=False
    )
    target_tokens = [tok.decode([i]) for i in target_ids]

    print(label)
    print("Input :", item["input"])
    print("Target:", item["target"])
    print("Target IDs:", target_ids)
    print("Target tokens:", target_tokens)
    print("Target length:", len(target_ids))
    print()

A
Input : The user tests the model.
Target: user tests model
Target IDs: [846, 6813, 1558]
Target tokens: ['user', ' tests', ' model']
Target length: 3

B
Input : The model tests the user.
Target: model tests user
Target IDs: [2448, 6813, 1156]
Target tokens: ['model', ' tests', ' user']
Target length: 3



## Single-token J-Lens pilot

First test whether standard single-token J-Lens can distinguish the two structural interpretations on Pair 1.

In [4]:
import inspect

print(inspect.signature(lens.readout))

(model, tok, prompt, layers=None, positions=None, use_jacobian=True, max_len=1024)


## Pair 1 single-token check

Compare support for `user` and `model` across selected late layers.

In [6]:
layers = [20, 24, 25, 26, 27, 28]

for label, item in pair1_targets.items():
    jl, _, _ = lens.readout(
        model,
        tok,
        item["input"],
        layers=layers,
        positions=[-1],
        use_jacobian=True,
    )

    first_target_id = tok.encode(
        item["target"],
        add_special_tokens=False
    )[0]

    first_target_token = tok.decode([first_target_id])

    print("=" * 70)
    print(label)
    print("Input:", item["input"])
    print("First target token:", repr(first_target_token))
    print()

    for layer in layers:
        logits = jl[layer][0]

        rank = (
            (logits > logits[first_target_id])
            .sum()
            .item()
            + 1
        )

        top5_ids = logits.topk(5).indices.tolist()
        top5_tokens = [tok.decode([i]) for i in top5_ids]

        print(
            f"L{layer:>2} | "
            f"target rank={rank:<6} | "
            f"top5={top5_tokens}"
        )

    print()

A
Input: The user tests the model.
First target token: 'user'

L20 | target rank=833    | top5=[':', '.', ':\\', '\\n', '?']
L24 | target rank=386    | top5=[' Tests', ' Testing', ' Model', ' Test', ' tests']
L25 | target rank=338    | top5=[' They', ' Model', ' If', ' model', ' Testing']
L26 | target rank=541    | top5=[' They', ' Testing', ' Tests', ' If', ' Test']
L27 | target rank=217    | top5=[' Tests', ' Testing', ' tests', ' They', ' Test']
L28 | target rank=364    | top5=[' They', ' The', ' If', ' Tests', ' Testing']

B
Input: The model tests the user.
First target token: 'model'

L20 | target rank=7178   | top5=['.', ':', ' If', ' test', ' tests']
L24 | target rank=255    | top5=[' If', ' Users', ' User', ' Tests', ' Test']
L25 | target rank=216    | top5=[' If', ' Users', ' User', ' user', ' Does']
L26 | target rank=450    | top5=[' If', ' Users', ' Does', ' User', ' When']
L27 | target rank=269    | top5=[' If', ' User', ' Users', ' Does', ' When']
L28 | target rank=508    

In [7]:
for text in [
    "user",
    " user",
    "model",
    " model",
    "user tests model",
    " user tests model",
]:
    ids = tok.encode(text, add_special_tokens=False)
    tokens = [tok.decode([i]) for i in ids]

    print(repr(text))
    print("IDs:", ids)
    print("Tokens:", [repr(t) for t in tokens])
    print()

'user'
IDs: [846]
Tokens: ["'user'"]

' user'
IDs: [1156]
Tokens: ["' user'"]

'model'
IDs: [2448]
Tokens: ["'model'"]

' model'
IDs: [1558]
Tokens: ["' model'"]

'user tests model'
IDs: [846, 6813, 1558]
Tokens: ["'user'", "' tests'", "' model'"]

' user tests model'
IDs: [1156, 6813, 1558]
Tokens: ["' user'", "' tests'", "' model'"]



In [8]:
pair1_targets_spaced = {
    "A": {
        "input": "The user tests the model.",
        "target_first_token": " user",
    },
    "B": {
        "input": "The model tests the user.",
        "target_first_token": " model",
    },
}

layers = [20, 24, 25, 26, 27, 28]

for label, item in pair1_targets_spaced.items():
    jl, _, _ = lens.readout(
        model,
        tok,
        item["input"],
        layers=layers,
        positions=[-1],
        use_jacobian=True,
    )

    target_id = tok.encode(
        item["target_first_token"],
        add_special_tokens=False
    )[0]

    print("=" * 70)
    print(label)
    print("Input:", item["input"])
    print("Target token:", repr(tok.decode([target_id])))
    print()

    for layer in layers:
        logits = jl[layer][0]

        rank = (
            (logits > logits[target_id])
            .sum()
            .item()
            + 1
        )

        top5_ids = logits.topk(5).indices.tolist()
        top5_tokens = [tok.decode([i]) for i in top5_ids]

        print(
            f"L{layer:>2} | "
            f"target rank={rank:<6} | "
            f"top5={top5_tokens}"
        )

    print()

A
Input: The user tests the model.
Target token: ' user'

L20 | target rank=137    | top5=[':', '.', ':\\', '\\n', '?']
L24 | target rank=62     | top5=[' Tests', ' Testing', ' Model', ' Test', ' tests']
L25 | target rank=64     | top5=[' They', ' Model', ' If', ' model', ' Testing']
L26 | target rank=131    | top5=[' They', ' Testing', ' Tests', ' If', ' Test']
L27 | target rank=32     | top5=[' Tests', ' Testing', ' tests', ' They', ' Test']
L28 | target rank=62     | top5=[' They', ' The', ' If', ' Tests', ' Testing']

B
Input: The model tests the user.
Target token: ' model'

L20 | target rank=705    | top5=['.', ':', ' If', ' test', ' tests']
L24 | target rank=45     | top5=[' If', ' Users', ' User', ' Tests', ' Test']
L25 | target rank=40     | top5=[' If', ' Users', ' User', ' user', ' Does']
L26 | target rank=139    | top5=[' If', ' Users', ' Does', ' User', ' When']
L27 | target rank=74     | top5=[' If', ' User', ' Users', ' Does', ' When']
L28 | target rank=156    | top5=[' 

## Pairwise single-token score

Compare the first tokens of the two candidate interpretations directly.

`S_margin = JLens(correct) - JLens(incorrect)`

This is an exploratory single-token check rather than the final structural metric.

In [9]:
pair1_s = {
    "A": {
        "input": "The user tests the model.",
        "correct": " user",
        "incorrect": " model",
    },
    "B": {
        "input": "The model tests the user.",
        "correct": " model",
        "incorrect": " user",
    },
}

layers = [20, 24, 25, 26, 27, 28]

for label, item in pair1_s.items():
    jl, _, _ = lens.readout(
        model,
        tok,
        item["input"],
        layers=layers,
        positions=[-1],
        use_jacobian=True,
    )

    correct_id = tok.encode(item["correct"], add_special_tokens=False)[0]
    incorrect_id = tok.encode(item["incorrect"], add_special_tokens=False)[0]

    print("=" * 72)
    print(label, "|", item["input"])
    print(
        "correct:", repr(tok.decode([correct_id])),
        "| incorrect:", repr(tok.decode([incorrect_id]))
    )

    for layer in layers:
        logits = jl[layer][0]

        correct_score = logits[correct_id].item()
        incorrect_score = logits[incorrect_id].item()
        margin = correct_score - incorrect_score

        winner = "CORRECT" if margin > 0 else "WRONG"

        print(
            f"L{layer:>2} | "
            f"S_margin={margin:>8.3f} | "
            f"winner={winner}"
        )

    print()

A | The user tests the model.
correct: ' user' | incorrect: ' model'
L20 | S_margin=   0.312 | winner=CORRECT
L24 | S_margin=  -3.062 | winner=WRONG
L25 | S_margin=  -3.375 | winner=WRONG
L26 | S_margin=  -2.750 | winner=WRONG
L27 | S_margin=  -1.688 | winner=WRONG
L28 | S_margin=  -1.562 | winner=WRONG

B | The model tests the user.
correct: ' model' | incorrect: ' user'
L20 | S_margin=  -1.938 | winner=WRONG
L24 | S_margin=  -3.625 | winner=WRONG
L25 | S_margin=  -3.875 | winner=WRONG
L26 | S_margin=  -4.562 | winner=WRONG
L27 | S_margin=  -4.062 | winner=WRONG
L28 | S_margin=  -4.062 | winner=WRONG



## Natural first-token clues

Inspect the tokens J-Lens naturally surfaces instead of forcing a hand-written first token.

In [10]:
pilot_input = "The user tests the model."
layers = [20, 24, 25, 26, 27, 28]

jl, _, _ = lens.readout(
    model,
    tok,
    pilot_input,
    layers=layers,
    positions=[-1],
    use_jacobian=True,
)

for layer in layers:
    logits = jl[layer][0]

    top_ids = logits.topk(10).indices.tolist()
    top_tokens = [tok.decode([i]) for i in top_ids]

    print(f"L{layer:>2}:", top_tokens)

L20: [':', '.', ':\\', '\\n', '?', ' If', ' After', ' Test', ' The', ' test']
L24: [' Tests', ' Testing', ' Model', ' Test', ' tests', ' Models', ' If', ' model', ' test', ' Results']
L25: [' They', ' Model', ' If', ' model', ' Testing', ' Models', ' Tests', ' Results', ' Users', ' tests']
L26: [' They', ' Testing', ' Tests', ' If', ' Test', ' Model', ' Results', ' tests', ' testing', ' After']
L27: [' Tests', ' Testing', ' tests', ' They', ' Test', ' If', ' Model', ' testing', ' test', ' Results']
L28: [' They', ' The', ' If', ' Tests', ' Testing', ' Model', ' This', ' After', ' When', ' Does']


## Inspect multi-token utilities

Check whether the J-Lens repository already contains utilities for continuation or multi-token readout.

In [11]:
import inspect

print("jlens module functions:")
for name, obj in inspect.getmembers(jlens):
    if inspect.isfunction(obj) or inspect.isclass(obj):
        print(name)

jlens module functions:
JLens
encode
fit
jacobian_for_prompt
load_model
load_wikitext
record_residuals
unembed


In [12]:
print("JLens methods:")

for name, obj in inspect.getmembers(jlens.JLens):
    if not name.startswith("_"):
        print(name)

JLens methods:
decompose
from_pretrained
load
readout
save
transport
vectors


## Inspect J-Lens vector utilities

Inspect the available vector methods before building the multi-token measurement.

In [13]:
for method_name in ["vectors", "transport", "decompose"]:
    method = getattr(jlens.JLens, method_name)

    print("=" * 80)
    print(method_name)
    print("Signature:")
    print(inspect.signature(method))
    print()

    try:
        print(inspect.getsource(method))
    except Exception as e:
        print("Could not show source:", e)

    print()

vectors
Signature:
(self, model, layer, token_ids)

    def vectors(self, model, layer, token_ids):
        """J-lens vectors: rows of W_U diag(g) J_l for the given tokens, [n, d].

        (Paper: rows of W_U J_l; we fold in the final RMSNorm's elementwise
        scale g so the vectors match the actual readout path.)
        """
        w = model.lm_head.weight[list(token_ids)].float().cpu()  # [n, d]
        g = model.model.norm.weight.float().cpu()
        return (w * g) @ self.J[layer]


transport
Signature:
(self, h, layer)

    def transport(self, h, layer):
        J = self.J[layer].to(device=h.device, dtype=torch.float32)
        return h.float() @ J.T


decompose
Signature:
(self, model, layer, h, k=16, n_candidates=512)

    def decompose(self, model, layer, h, k=16, n_candidates=512):
        """Sparse nonnegative pursuit of h against this layer's lens vectors.

        Greedy: candidates are the top-n_candidates tokens by lens logit; each
        step adds the atom with th

## Experimental conditions

I considered four measurements during development:

- **S**: exploratory single-token J-Lens readout
- **L0**: context-only reference
- **L1**: first-token-conditioned LM continuation
- **M**: activation-side structural alignment

S was later dropped as the main structural metric because many of the tested relations depend on multiple tokens and their roles.

The main held-out comparison is between L1 and M using a common directional success criterion. Their raw scores are not subtracted because they are measured in different units.

## L1 continuation pilot

Give the frozen LM the correct first candidate token and score only the remaining tail.

This tests how much of the multi-token distinction ordinary continuation can recover once the first token is known.

In [14]:
import torch

@torch.no_grad()
def continuation_logprob(head, tail):
    head_ids = tok.encode(head, add_special_tokens=False)
    tail_ids = tok.encode(tail, add_special_tokens=False)

    full_ids = head_ids + tail_ids

    input_ids = torch.tensor(
        [full_ids],
        device=model.device
    )

    outputs = model(input_ids=input_ids)
    log_probs = torch.log_softmax(outputs.logits, dim=-1)

    token_logps = []

    # token at position i is predicted from logits at i-1
    for i in range(len(head_ids), len(full_ids)):
        target_id = full_ids[i]
        lp = log_probs[0, i - 1, target_id].item()
        token_logps.append(lp)

    return {
        "head": head,
        "tail": tail,
        "tail_tokens": [tok.decode([i]) for i in tail_ids],
        "token_logprobs": token_logps,
        "sum_logprob": sum(token_logps),
        "mean_logprob": sum(token_logps) / len(token_logps),
    }


l1_a = continuation_logprob(
    head=" user",
    tail=" tests model",
)

l1_b = continuation_logprob(
    head=" model",
    tail=" tests user",
)

for label, result in [("A", l1_a), ("B", l1_b)]:
    print("=" * 60)
    print(label)
    print("Head:", repr(result["head"]))
    print("Tail:", repr(result["tail"]))
    print("Tail tokens:", result["tail_tokens"])
    print("Token log-probs:", result["token_logprobs"])
    print("Sum log-prob:", result["sum_logprob"])
    print("Mean log-prob:", result["mean_logprob"])

A
Head: ' user'
Tail: ' tests model'
Tail tokens: [' tests', ' model']
Token log-probs: [-8.3125, -8.75]
Sum log-prob: -17.0625
Mean log-prob: -8.53125
B
Head: ' model'
Tail: ' tests user'
Tail tokens: [' tests', ' user']
Token log-probs: [-8.3125, -10.0625]
Sum log-prob: -18.375
Mean log-prob: -9.1875


## Neutral L1 scaffold

Use the same fixed `Relation:` scaffold for every candidate so the continuation baseline starts from a consistent context.

In [15]:
@torch.no_grad()
def scaffolded_continuation_logprob(head, tail, scaffold="Relation:"):
    prefix_ids = tok.encode(scaffold + head, add_special_tokens=False)
    tail_ids = tok.encode(tail, add_special_tokens=False)

    full_ids = prefix_ids + tail_ids

    input_ids = torch.tensor(
        [full_ids],
        device=model.device
    )

    outputs = model(input_ids=input_ids)
    log_probs = torch.log_softmax(outputs.logits, dim=-1)

    token_logps = []

    for i in range(len(prefix_ids), len(full_ids)):
        target_id = full_ids[i]
        lp = log_probs[0, i - 1, target_id].item()
        token_logps.append(lp)

    return {
        "head": head,
        "tail": tail,
        "tail_tokens": [tok.decode([i]) for i in tail_ids],
        "token_logprobs": token_logps,
        "sum_logprob": sum(token_logps),
        "mean_logprob": sum(token_logps) / len(token_logps),
    }


l1_a_scaffold = scaffolded_continuation_logprob(
    head=" user",
    tail=" tests model",
)

l1_b_scaffold = scaffolded_continuation_logprob(
    head=" model",
    tail=" tests user",
)

for label, result in [
    ("A", l1_a_scaffold),
    ("B", l1_b_scaffold)
]:
    print("=" * 60)
    print(label)
    print("Head:", repr(result["head"]))
    print("Tail:", repr(result["tail"]))
    print("Tail tokens:", result["tail_tokens"])
    print("Token log-probs:", result["token_logprobs"])
    print("Mean log-prob:", result["mean_logprob"])

A
Head: ' user'
Tail: ' tests model'
Tail tokens: [' tests', ' model']
Token log-probs: [-11.75, -6.65625]
Mean log-prob: -9.203125
B
Head: ' model'
Tail: ' tests user'
Tail tokens: [' tests', ' user']
Token log-probs: [-9.0, -11.0625]
Mean log-prob: -10.03125


## Non-copying source-target pilot

Separate the source wording from the candidate wording so the measurement cannot succeed through direct phrase copying.

For Pair 1, the source describes the relation using different words while the candidates remain:

`user tests model`  
`model tests user`

In [16]:
pair1_structural_pilot = {
    "A": {
        "source": (
            "Alice is the user. Bob is the model. "
            "Alice evaluates Bob's behavior."
        ),
        "target": "user tests model",
    },
    "B": {
        "source": (
            "Alice is the user. Bob is the model. "
            "Bob evaluates Alice's behavior."
        ),
        "target": "model tests user",
    },
}

for label, item in pair1_structural_pilot.items():
    source_ids = tok.encode(item["source"], add_special_tokens=False)
    target_ids = tok.encode(item["target"], add_special_tokens=False)

    print("=" * 70)
    print(label)
    print("Source:", item["source"])
    print("Target:", item["target"])
    print("Source length:", len(source_ids))
    print("Target tokens:", [tok.decode([i]) for i in target_ids])
    print("Target length:", len(target_ids))

A
Source: Alice is the user. Bob is the model. Alice evaluates Bob's behavior.
Target: user tests model
Source length: 16
Target tokens: ['user', ' tests', ' model']
Target length: 3
B
Source: Alice is the user. Bob is the model. Bob evaluates Alice's behavior.
Target: model tests user
Source length: 16
Target tokens: ['model', ' tests', ' user']
Target length: 3


## Behavioral sanity check

Before testing internal representations, check that the frozen model can distinguish the intended relation from the non-copying source context.

In [17]:
for label, item in pair1_structural_pilot.items():

    prompt = (
        item["source"]
        + "\nExpress the relation using exactly three words in the form:"
        + "\nrole action role"
        + "\nRelation:"
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
    )

    generated = tok.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    print("=" * 70)
    print(label)
    print("Expected:", item["target"])
    print("Generated:", repr(generated))

A
Expected: user tests model
Generated: '\nrole action role\n\n<think>\nThinking'
B
Expected: model tests user
Generated: '\n<think>\nThinking Process:\n\n1'


## Pairwise behavioral check

Instead of relying on free generation, directly compare the likelihood of the two candidate interpretations.

In [18]:
import torch

@torch.no_grad()
def candidate_logprob(source, candidate):
    prefix = (
        source
        + "\n\nThe relation can be summarized as:"
    )

    prefix_ids = tok.encode(prefix, add_special_tokens=False)
    candidate_ids = tok.encode(
        " " + candidate,
        add_special_tokens=False
    )

    full_ids = prefix_ids + candidate_ids

    input_ids = torch.tensor(
        [full_ids],
        device=model.device
    )

    outputs = model(input_ids=input_ids)
    log_probs = torch.log_softmax(outputs.logits, dim=-1)

    token_logps = []

    for i in range(len(prefix_ids), len(full_ids)):
        target_id = full_ids[i]
        token_logps.append(
            log_probs[0, i - 1, target_id].item()
        )

    return sum(token_logps) / len(token_logps)


behavioral_tests = {
    "A": {
        "source": pair1_structural_pilot["A"]["source"],
        "correct": "user tests model",
        "wrong": "model tests user",
    },
    "B": {
        "source": pair1_structural_pilot["B"]["source"],
        "correct": "model tests user",
        "wrong": "user tests model",
    },
}

for label, item in behavioral_tests.items():

    correct_score = candidate_logprob(
        item["source"],
        item["correct"]
    )

    wrong_score = candidate_logprob(
        item["source"],
        item["wrong"]
    )

    margin = correct_score - wrong_score

    print("=" * 70)
    print(label)
    print("Correct:", item["correct"])
    print("Wrong:  ", item["wrong"])
    print("Correct mean log-prob:", correct_score)
    print("Wrong mean log-prob:  ", wrong_score)
    print("Margin:", margin)
    print(
        "Winner:",
        "CORRECT" if margin > 0 else "WRONG"
    )

A
Correct: user tests model
Wrong:   model tests user
Correct mean log-prob: -5.283854166666667
Wrong mean log-prob:   -7.1171875
Margin: 1.833333333333333
Winner: CORRECT
B
Correct: model tests user
Wrong:   user tests model
Correct mean log-prob: -6.364583333333333
Wrong mean log-prob:   -5.658854166666667
Margin: -0.7057291666666661
Winner: WRONG


## Prior-adjusted check

Check whether candidate preferences are partly explained by the LM's general phrase prior rather than the source relation itself.

In [19]:
neutral_source = "Consider the relation between two roles."

for label, item in behavioral_tests.items():

    correct_cond = candidate_logprob(
        item["source"],
        item["correct"]
    )

    wrong_cond = candidate_logprob(
        item["source"],
        item["wrong"]
    )

    correct_base = candidate_logprob(
        neutral_source,
        item["correct"]
    )

    wrong_base = candidate_logprob(
        neutral_source,
        item["wrong"]
    )

    correct_gain = correct_cond - correct_base
    wrong_gain = wrong_cond - wrong_base

    adjusted_margin = correct_gain - wrong_gain

    print("=" * 70)
    print(label)

    print("Correct candidate:", item["correct"])
    print("  conditional:", correct_cond)
    print("  baseline:   ", correct_base)
    print("  context gain:", correct_gain)

    print()

    print("Wrong candidate:", item["wrong"])
    print("  conditional:", wrong_cond)
    print("  baseline:   ", wrong_base)
    print("  context gain:", wrong_gain)

    print()

    print("PRIOR-ADJUSTED MARGIN:", adjusted_margin)
    print(
        "Winner:",
        "CORRECT" if adjusted_margin > 0 else "WRONG"
    )

A
Correct candidate: user tests model
  conditional: -5.283854166666667
  baseline:    -8.791666666666666
  context gain: 3.507812499999999

Wrong candidate: model tests user
  conditional: -7.1171875
  baseline:    -10.791666666666666
  context gain: 3.674479166666666

PRIOR-ADJUSTED MARGIN: -0.16666666666666696
Winner: WRONG
B
Correct candidate: model tests user
  conditional: -6.364583333333333
  baseline:    -10.791666666666666
  context gain: 4.427083333333333

Wrong candidate: user tests model
  conditional: -5.658854166666667
  baseline:    -8.791666666666666
  context gain: 3.132812499999999

PRIOR-ADJUSTED MARGIN: 1.294270833333334
Winner: CORRECT


## Paired structural shift

Measure how the preference between C1 and C2 changes when the source relation is reversed.

A positive shift means the source reversal moves candidate preference in the expected direction.

In [20]:
c1 = "user tests model"
c2 = "model tests user"

source_a = pair1_structural_pilot["A"]["source"]
source_b = pair1_structural_pilot["B"]["source"]

a_c1 = candidate_logprob(source_a, c1)
a_c2 = candidate_logprob(source_a, c2)

b_c1 = candidate_logprob(source_b, c1)
b_c2 = candidate_logprob(source_b, c2)

log_odds_a = a_c1 - a_c2
log_odds_b = b_c1 - b_c2

paired_shift = log_odds_a - log_odds_b

print("Source A log-odds (C1 - C2):", log_odds_a)
print("Source B log-odds (C1 - C2):", log_odds_b)
print()
print("Paired structural shift:", paired_shift)
print(
    "Expected-direction shift:",
    paired_shift > 0
)

Source A log-odds (C1 - C2): 1.833333333333333
Source B log-odds (C1 - C2): 0.7057291666666661

Paired structural shift: 1.127604166666667
Expected-direction shift: True


## L1: first-token-conditioned continuation

For each candidate, provide its correct first token and score only the remaining tokens.

The key measurement is whether reversing the source relation shifts the C1 versus C2 preference in the expected direction.

In [21]:
@torch.no_grad()
def source_head_tail_logprob(source, head, tail):
    prefix = (
        source
        + "\n\nThe relation can be summarized as:"
        + head
    )

    prefix_ids = tok.encode(prefix, add_special_tokens=False)
    tail_ids = tok.encode(tail, add_special_tokens=False)

    full_ids = prefix_ids + tail_ids

    input_ids = torch.tensor(
        [full_ids],
        device=model.device
    )

    outputs = model(input_ids=input_ids)
    log_probs = torch.log_softmax(outputs.logits, dim=-1)

    token_logps = []

    for i in range(len(prefix_ids), len(full_ids)):
        target_id = full_ids[i]
        token_logps.append(
            log_probs[0, i - 1, target_id].item()
        )

    return sum(token_logps) / len(token_logps)


# Candidate 1: user tests model
# Candidate 2: model tests user

a_c1_l1 = source_head_tail_logprob(
    source_a, " user", " tests model"
)

a_c2_l1 = source_head_tail_logprob(
    source_a, " model", " tests user"
)

b_c1_l1 = source_head_tail_logprob(
    source_b, " user", " tests model"
)

b_c2_l1 = source_head_tail_logprob(
    source_b, " model", " tests user"
)

l1_log_odds_a = a_c1_l1 - a_c2_l1
l1_log_odds_b = b_c1_l1 - b_c2_l1

l1_paired_shift = l1_log_odds_a - l1_log_odds_b

print("L1 Source A log-odds (C1 - C2):", l1_log_odds_a)
print("L1 Source B log-odds (C1 - C2):", l1_log_odds_b)
print()
print("L1 paired structural shift:", l1_paired_shift)
print(
    "Expected-direction shift:",
    l1_paired_shift > 0
)

L1 Source A log-odds (C1 - C2): 1.203125
L1 Source B log-odds (C1 - C2): -0.73828125

L1 paired structural shift: 1.94140625
Expected-direction shift: True


## Multi-token concept vectors

Represent each candidate inside the same fixed carrier prompt and extract post-concept hidden states.

This is a lightweight activation-side construction, not a full calibrated multi-token J-Lens.

In [22]:
carrier_template = (
    "Remember the following concept in your mind.\n"
    "Concept: {}. Fact: The capital of France is Paris."
)

for concept in [
    "user tests model",
    "model tests user",
]:
    carrier = carrier_template.format(concept)

    ids = tok.encode(carrier, add_special_tokens=False)
    tokens = [tok.decode([i]) for i in ids]

    concept_ids = tok.encode(concept, add_special_tokens=False)

    print("=" * 70)
    print("Concept:", concept)
    print("Carrier:", carrier)
    print("Carrier length:", len(ids))
    print("Concept token IDs:", concept_ids)
    print("Concept tokens:", [tok.decode([i]) for i in concept_ids])
    print()

Concept: user tests model
Carrier: Remember the following concept in your mind.
Concept: user tests model. Fact: The capital of France is Paris.
Carrier length: 24
Concept token IDs: [846, 6813, 1558]
Concept tokens: ['user', ' tests', ' model']

Concept: model tests user
Carrier: Remember the following concept in your mind.
Concept: model tests user. Fact: The capital of France is Paris.
Carrier length: 24
Concept token IDs: [2448, 6813, 1156]
Concept tokens: ['model', ' tests', ' user']



## Locate the concept span

Identify the exact candidate token positions inside the carrier prompt before extracting hidden states.

In [23]:
carrier_prefix = (
    "Remember the following concept in your mind.\n"
    "Concept:"
)

carrier_suffix = ". Fact: The capital of France is Paris."

for concept in [
    "user tests model",
    "model tests user",
]:
    carrier = carrier_prefix + " " + concept + carrier_suffix

    carrier_ids = tok.encode(carrier, add_special_tokens=False)
    concept_ids_in_context = tok.encode(
        " " + concept,
        add_special_tokens=False
    )

    # Find the exact concept token sequence inside the carrier
    matches = []

    for i in range(
        len(carrier_ids) - len(concept_ids_in_context) + 1
    ):
        if (
            carrier_ids[i:i + len(concept_ids_in_context)]
            == concept_ids_in_context
        ):
            matches.append(i)

    print("=" * 70)
    print("Concept:", concept)
    print(
        "Concept tokens in context:",
        [tok.decode([i]) for i in concept_ids_in_context]
    )
    print("Match positions:", matches)

    if len(matches) == 1:
        start = matches[0]
        end = start + len(concept_ids_in_context) - 1

        print("Concept span:", start, "to", end)
        print("First post-concept position:", end + 1)
        print(
            "Decoded span:",
            repr(tok.decode(carrier_ids[start:end + 1]))
        )

    print()

Concept: user tests model
Concept tokens in context: [' user', ' tests', ' model']
Match positions: [11]
Concept span: 11 to 13
First post-concept position: 14
Decoded span: ' user tests model'

Concept: model tests user
Concept tokens in context: [' model', ' tests', ' user']
Match positions: [11]
Concept span: 11 to 13
First post-concept position: 14
Decoded span: ' model tests user'



## Capture post-concept states

Record residual-stream activations immediately after each candidate concept.

In [24]:
carrier_layer = 24
post_start = 14
post_window = 4

concept_hidden_states = {}

for concept in [
    "user tests model",
    "model tests user",
]:
    carrier = carrier_prefix + " " + concept + carrier_suffix

    ids = jlens.encode(model, tok, carrier)

    with jlens.record_residuals(model, [carrier_layer]) as rec:
        model.model(
            input_ids=ids,
            use_cache=False
        )

    h = rec.acts[carrier_layer][0].float().cpu()

    post_h = h[
        post_start : post_start + post_window
    ]

    concept_hidden_states[concept] = post_h

    token_ids = ids[0].tolist()

    print("=" * 70)
    print("Concept:", concept)
    print("Full hidden-state shape:", tuple(h.shape))
    print("Post-concept shape:", tuple(post_h.shape))
    print()

    for pos in range(post_start, post_start + post_window):
        print(
            f"position {pos}:",
            repr(tok.decode([token_ids[pos]]))
        )

    print()

Concept: user tests model
Full hidden-state shape: (24, 2560)
Post-concept shape: (4, 2560)

position 14: '.'
position 15: ' Fact'
position 16: ':'
position 17: ' The'

Concept: model tests user
Full hidden-state shape: (24, 2560)
Post-concept shape: (4, 2560)

position 14: '.'
position 15: ' Fact'
position 16: ':'
position 17: ' The'



## Lightweight concept vectors

Mean-pool four post-concept residual states to obtain one vector for each candidate.

This is a simple pilot representation rather than a full multi-token recovery method.

In [25]:
import torch
import torch.nn.functional as F

concept_vectors = {}

for concept, post_h in concept_hidden_states.items():
    # Lightweight pilot: uniform mean pooling
    v = post_h.mean(dim=0)

    concept_vectors[concept] = v

    print(
        concept,
        "| vector shape:", tuple(v.shape),
        "| norm:", v.norm().item()
    )

v_c1 = concept_vectors["user tests model"]
v_c2 = concept_vectors["model tests user"]

cosine_between_candidates = F.cosine_similarity(
    v_c1.unsqueeze(0),
    v_c2.unsqueeze(0)
).item()

delta_norm = (v_c1 - v_c2).norm().item()

print()
print("Cosine similarity between candidate vectors:", cosine_between_candidates)
print("Difference-vector norm:", delta_norm)

user tests model | vector shape: (2560,) | norm: 26.2379207611084
model tests user | vector shape: (2560,) | norm: 25.884624481201172

Cosine similarity between candidate vectors: 0.9818916916847229
Difference-vector norm: 4.972034931182861


## Initial activation-side score

Compare source activations with the two candidate concept vectors as an initial structural diagnostic.

In [26]:
source_hidden_states = {}

for label, item in pair1_structural_pilot.items():

    ids = jlens.encode(
        model,
        tok,
        item["source"]
    )

    with jlens.record_residuals(
        model,
        [carrier_layer]
    ) as rec:

        model.model(
            input_ids=ids,
            use_cache=False
        )

    h = rec.acts[carrier_layer][0].float().cpu()

    # Final source position
    source_h = h[-1]

    source_hidden_states[label] = source_h

    print(
        label,
        "| source activation shape:",
        tuple(source_h.shape),
        "| norm:",
        source_h.norm().item()
    )

A | source activation shape: (2560,) | norm: 33.8894157409668
B | source activation shape: (2560,) | norm: 33.14493179321289


In [27]:
def cos(a, b):
    return F.cosine_similarity(
        a.unsqueeze(0),
        b.unsqueeze(0)
    ).item()


source_a_h = source_hidden_states["A"]
source_b_h = source_hidden_states["B"]

c1 = concept_vectors["user tests model"]
c2 = concept_vectors["model tests user"]


# Source A
a_c1_m = cos(source_a_h, c1)
a_c2_m = cos(source_a_h, c2)
m_pref_a = a_c1_m - a_c2_m

# Source B
b_c1_m = cos(source_b_h, c1)
b_c2_m = cos(source_b_h, c2)
m_pref_b = b_c1_m - b_c2_m

# Minimal-pair structural shift
m_paired_shift = m_pref_a - m_pref_b


print("SOURCE A")
print("sim(A, user tests model):", a_c1_m)
print("sim(A, model tests user):", a_c2_m)
print("preference C1-C2:", m_pref_a)

print()
print("SOURCE B")
print("sim(B, user tests model):", b_c1_m)
print("sim(B, model tests user):", b_c2_m)
print("preference C1-C2:", m_pref_b)

print()
print("LIGHTWEIGHT M PAIRED SHIFT:", m_paired_shift)
print("Expected-direction shift:", m_paired_shift > 0)

SOURCE A
sim(A, user tests model): 0.6650775074958801
sim(A, model tests user): 0.6507315635681152
preference C1-C2: 0.014345943927764893

SOURCE B
sim(B, user tests model): 0.6669729948043823
sim(B, model tests user): 0.6558245420455933
preference C1-C2: 0.011148452758789062

LIGHTWEIGHT M PAIRED SHIFT: 0.00319749116897583
Expected-direction shift: True


## Shared-carrier check

Check whether the fixed carrier prompt contributes a large common component to both candidate vectors.

In [28]:
# Shared component of the two candidate vectors
shared_v = (v_c1 + v_c2) / 2

# Candidate-specific residuals
c1_centered = v_c1 - shared_v
c2_centered = v_c2 - shared_v

print("Original cosine:", cos(v_c1, v_c2))
print("Centered C1 norm:", c1_centered.norm().item())
print("Centered C2 norm:", c2_centered.norm().item())
print(
    "Centered cosine:",
    cos(c1_centered, c2_centered)
)

Original cosine: 0.9818916916847229
Centered C1 norm: 2.4860174655914307
Centered C2 norm: 2.4860174655914307
Centered cosine: -1.0000004768371582


## Structural difference-direction diagnostic

Compare two changes in activation space:

`source_difference = h_A - h_B`

`concept_difference = v_C1 - v_C2`

M is the cosine alignment between these directions.

A positive value means the source relation reversal moves activation space in the same direction that separates the two candidate concepts.

In [30]:
source_difference = (
    source_hidden_states["A"]
    - source_hidden_states["B"]
)

concept_difference = (
    concept_vectors["user tests model"]
    - concept_vectors["model tests user"]
)

difference_alignment = cos(
    source_difference,
    concept_difference
)

print("Source-difference norm:", source_difference.norm().item())
print("Concept-difference norm:", concept_difference.norm().item())
print()
print(
    "Source ↔ concept difference alignment:",
    difference_alignment
)
print(
    "Expected-direction alignment:",
    difference_alignment > 0
)

Source-difference norm: 7.457942962646484
Concept-difference norm: 4.972034931182861

Source ↔ concept difference alignment: 0.08554947376251221
Expected-direction alignment: True


## Layer sweep

Measure the same structural alignment across the predefined late-layer set:

`[20, 22, 24, 25, 26, 27, 28]`

In [31]:
sweep_layers = [20, 22, 24, 25, 26, 27, 28]

layer_results = []

for layer in sweep_layers:

    # --------------------------------
    # 1. Build candidate concept vectors
    # --------------------------------
    concept_vecs_layer = {}

    for concept in [
        "user tests model",
        "model tests user",
    ]:
        carrier = carrier_prefix + " " + concept + carrier_suffix

        ids = jlens.encode(model, tok, carrier)

        with jlens.record_residuals(model, [layer]) as rec:
            model.model(
                input_ids=ids,
                use_cache=False
            )

        h = rec.acts[layer][0].float().cpu()

        post_h = h[
            post_start : post_start + post_window
        ]

        concept_vecs_layer[concept] = post_h.mean(dim=0)

    # --------------------------------
    # 2. Get source activations
    # --------------------------------
    source_vecs_layer = {}

    for label, item in pair1_structural_pilot.items():

        ids = jlens.encode(
            model,
            tok,
            item["source"]
        )

        with jlens.record_residuals(model, [layer]) as rec:
            model.model(
                input_ids=ids,
                use_cache=False
            )

        h = rec.acts[layer][0].float().cpu()

        source_vecs_layer[label] = h[-1]

    # --------------------------------
    # 3. Difference directions
    # --------------------------------
    source_diff = (
        source_vecs_layer["A"]
        - source_vecs_layer["B"]
    )

    concept_diff = (
        concept_vecs_layer["user tests model"]
        - concept_vecs_layer["model tests user"]
    )

    alignment = cos(
        source_diff,
        concept_diff
    )

    layer_results.append(
        {
            "layer": layer,
            "alignment": alignment,
        }
    )

    print(
        f"L{layer:>2} | "
        f"alignment={alignment:+.4f} | "
        f"expected_direction={alignment > 0}"
    )

L20 | alignment=+0.0566 | expected_direction=True
L22 | alignment=+0.0744 | expected_direction=True
L24 | alignment=+0.0855 | expected_direction=True
L25 | alignment=+0.0990 | expected_direction=True
L26 | alignment=-0.1073 | expected_direction=False
L27 | alignment=+0.0862 | expected_direction=True
L28 | alignment=+0.1009 | expected_direction=True


## Aggregate layer summary

Summarize alignment across the full predefined layer set rather than selecting the best layer after seeing the result.

In [33]:
import numpy as np

alignments = np.array([
    r["alignment"]
    for r in layer_results
])

mean_alignment = alignments.mean()
median_alignment = np.median(alignments)
positive_fraction = (alignments > 0).mean()

print("Mean alignment:", mean_alignment)
print("Median alignment:", median_alignment)
print("Positive layers:", f"{(alignments > 0).sum()}/{len(alignments)}")
print("Positive fraction:", positive_fraction)

Mean alignment: 0.056479218282869885
Median alignment: 0.08554947376251221
Positive layers: 6/7
Positive fraction: 0.8571428571428571


## Freeze the pilot settings

All M design choices are frozen after Pair 1:

- carrier prompt
- four-token post-concept window
- mean pooling
- layers `[20, 22, 24, 25, 26, 27, 28]`
- final-token source residual
- source/concept difference-direction cosine

Pair 1 pilot result:

- mean alignment: +0.0565
- median alignment: +0.0855
- positive layers: 6/7

No settings are changed based on Pairs 2-15.

## Held-out structural set

Construct non-copying source contexts for Pairs 2-15 while keeping the estimator frozen.

In [34]:
structural_eval_set = [

    # --------------------------------------------------
    # Pair 2 — Role / Argument Binding
    # --------------------------------------------------
    {
        "pair_id": 2,
        "category": "role_argument_binding",
        "source_a": (
            "Alice is the doctor. Bob is the patient. "
            "Alice assists Bob during the visit."
        ),
        "source_b": (
            "Alice is the doctor. Bob is the patient. "
            "Bob assists Alice during the visit."
        ),
        "c1": "doctor helps patient",
        "c2": "patient helps doctor",
    },

    # --------------------------------------------------
    # Pair 3 — Modifier Attachment
    # --------------------------------------------------
    {
        "pair_id": 3,
        "category": "modifier_attachment",
        "source_a": (
            "There is a box and a key. The box is red. "
            "The box contains the key."
        ),
        "source_b": (
            "There is a box and a key. The key is red. "
            "The box contains the key."
        ),
        "c1": "red box contains key",
        "c2": "box contains red key",
    },

    # --------------------------------------------------
    # Pair 4 — Spatial Relation
    # --------------------------------------------------
    {
        "pair_id": 4,
        "category": "spatial_relation",
        "source_a": (
            "A cat and a dog are in the scene. "
            "The cat is positioned higher than the dog."
        ),
        "source_b": (
            "A cat and a dog are in the scene. "
            "The dog is positioned higher than the cat."
        ),
        "c1": "cat above dog",
        "c2": "dog above cat",
    },

    # --------------------------------------------------
    # Pair 5 — Comparative Relation
    # --------------------------------------------------
    {
        "pair_id": 5,
        "category": "comparative_relation",
        "source_a": (
            "There is a blue car and a red car. "
            "The blue car wins a speed comparison."
        ),
        "source_b": (
            "There is a blue car and a red car. "
            "The red car wins a speed comparison."
        ),
        "c1": "blue car faster than red car",
        "c2": "red car faster than blue car",
    },

    # --------------------------------------------------
    # Pair 6 — Role / Argument Binding
    # --------------------------------------------------
    {
        "pair_id": 6,
        "category": "role_argument_binding",
        "source_a": (
            "Alice is the teacher. Bob is the student. "
            "Alice walks behind Bob."
        ),
        "source_b": (
            "Alice is the teacher. Bob is the student. "
            "Bob walks behind Alice."
        ),
        "c1": "teacher follows student",
        "c2": "student follows teacher",
    },

    # --------------------------------------------------
    # Pair 7 — Modifier Attachment
    # --------------------------------------------------
    {
        "pair_id": 7,
        "category": "modifier_attachment",
        "source_a": (
            "There is a man and a bag. The man is old. "
            "The man carries the bag."
        ),
        "source_b": (
            "There is a man and a bag. The bag is old. "
            "The man carries the bag."
        ),
        "c1": "old man carries bag",
        "c2": "man carries old bag",
    },

    # --------------------------------------------------
    # Pair 8 — Modifier Attachment
    # --------------------------------------------------
    {
        "pair_id": 8,
        "category": "modifier_attachment",
        "source_a": (
            "There is a table and a book. The table is wooden. "
            "The table supports the book."
        ),
        "source_b": (
            "There is a table and a book. The book is wooden. "
            "The table supports the book."
        ),
        "c1": "wooden table holds book",
        "c2": "table holds wooden book",
    },

    # --------------------------------------------------
    # Pair 9 — Spatial Relation
    # --------------------------------------------------
    {
        "pair_id": 9,
        "category": "spatial_relation",
        "source_a": (
            "A lamp and a chair are in the room. "
            "The chair is positioned in front of the lamp."
        ),
        "source_b": (
            "A lamp and a chair are in the room. "
            "The lamp is positioned in front of the chair."
        ),
        "c1": "lamp behind chair",
        "c2": "chair behind lamp",
    },

    # --------------------------------------------------
    # Pair 10 — Spatial Relation
    # --------------------------------------------------
    {
        "pair_id": 10,
        "category": "spatial_relation",
        "source_a": (
            "A cup and a plate are stacked together. "
            "The plate is positioned higher than the cup."
        ),
        "source_b": (
            "A cup and a plate are stacked together. "
            "The cup is positioned higher than the plate."
        ),
        "c1": "cup under plate",
        "c2": "plate under cup",
    },

    # --------------------------------------------------
    # Pair 11 — Comparative Relation
    # --------------------------------------------------
    {
        "pair_id": 11,
        "category": "comparative_relation",
        "source_a": (
            "There is a green box and a yellow box. "
            "A scale shows the green box has greater weight."
        ),
        "source_b": (
            "There is a green box and a yellow box. "
            "A scale shows the yellow box has greater weight."
        ),
        "c1": "green box heavier than yellow box",
        "c2": "yellow box heavier than green box",
    },

    # --------------------------------------------------
    # Pair 12 — Comparative Relation
    # --------------------------------------------------
    {
        "pair_id": 12,
        "category": "comparative_relation",
        "source_a": (
            "A train and a bus are being measured. "
            "The train has greater length."
        ),
        "source_b": (
            "A train and a bus are being measured. "
            "The bus has greater length."
        ),
        "c1": "train longer than bus",
        "c2": "bus longer than train",
    },

    # --------------------------------------------------
    # Pair 13 — Quantifier Binding
    # --------------------------------------------------
    {
        "pair_id": 13,
        "category": "quantifier_binding",
        "source_a": (
            "In this scene, every dog is chasing at least one cat."
        ),
        "source_b": (
            "In this scene, at least one dog is chasing every cat."
        ),
        "c1": "all dogs chase some cats",
        "c2": "some dogs chase all cats",
    },

    # --------------------------------------------------
    # Pair 14 — Quantifier Binding
    # --------------------------------------------------
    {
        "pair_id": 14,
        "category": "quantifier_binding",
        "source_a": (
            "In this scene, every teacher assists at least one student."
        ),
        "source_b": (
            "In this scene, at least one teacher assists every student."
        ),
        "c1": "all teachers help some students",
        "c2": "some teachers help all students",
    },

    # --------------------------------------------------
    # Pair 15 — Quantifier Binding
    # --------------------------------------------------
    {
        "pair_id": 15,
        "category": "quantifier_binding",
        "source_a": (
            "In this scene, every artist admires at least one writer."
        ),
        "source_b": (
            "In this scene, at least one artist admires every writer."
        ),
        "c1": "all artists admire some writers",
        "c2": "some artists admire all writers",
    },
]

print("Held-out pairs:", len(structural_eval_set))

Held-out pairs: 14


## Held-out validation

Check source balance, candidate tokenization, token multiset, copying, and structural consistency before evaluation.

In [35]:
from collections import Counter

validation_rows = []

for item in structural_eval_set:

    source_a_ids = tok.encode(
        item["source_a"],
        add_special_tokens=False
    )

    source_b_ids = tok.encode(
        item["source_b"],
        add_special_tokens=False
    )

    c1_ids = tok.encode(
        item["c1"],
        add_special_tokens=False
    )

    c2_ids = tok.encode(
        item["c2"],
        add_special_tokens=False
    )

    same_source_length = (
        len(source_a_ids) == len(source_b_ids)
    )

    same_candidate_length = (
        len(c1_ids) == len(c2_ids)
    )

    same_candidate_multiset = (
        Counter(c1_ids) == Counter(c2_ids)
    )

    c1_copied_in_a = (
        item["c1"].lower()
        in item["source_a"].lower()
    )

    c2_copied_in_b = (
        item["c2"].lower()
        in item["source_b"].lower()
    )

    validation_rows.append({
        "pair_id": item["pair_id"],
        "category": item["category"],
        "source_len_a": len(source_a_ids),
        "source_len_b": len(source_b_ids),
        "same_source_length": same_source_length,
        "candidate_len_c1": len(c1_ids),
        "candidate_len_c2": len(c2_ids),
        "same_candidate_length": same_candidate_length,
        "same_candidate_multiset": same_candidate_multiset,
        "c1_copied_in_a": c1_copied_in_a,
        "c2_copied_in_b": c2_copied_in_b,
    })

for row in validation_rows:
    print(
        f"Pair {row['pair_id']:>2} | "
        f"{row['category']:<24} | "
        f"source={row['source_len_a']}/{row['source_len_b']} | "
        f"cand={row['candidate_len_c1']}/{row['candidate_len_c2']} | "
        f"same_source={row['same_source_length']} | "
        f"same_cand_len={row['same_candidate_length']} | "
        f"same_multiset={row['same_candidate_multiset']} | "
        f"copy={row['c1_copied_in_a'] or row['c2_copied_in_b']}"
    )

Pair  2 | role_argument_binding    | source=17/17 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=False | copy=False
Pair  3 | modifier_attachment      | source=19/19 | cand=4/4 | same_source=True | same_cand_len=True | same_multiset=False | copy=False
Pair  4 | spatial_relation         | source=19/19 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=False | copy=False
Pair  5 | comparative_relation     | source=18/18 | cand=6/6 | same_source=True | same_cand_len=True | same_multiset=False | copy=False
Pair  6 | role_argument_binding    | source=15/15 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=False | copy=False
Pair  7 | modifier_attachment      | source=19/19 | cand=4/4 | same_source=True | same_cand_len=True | same_multiset=False | copy=False
Pair  8 | modifier_attachment      | source=19/19 | cand=5/4 | same_source=True | same_cand_len=False | same_multiset=False | copy=False
Pair  9 | spatial_relation         | source=20/

## Tokenization validation fix

Candidate validation initially used standalone phrases.

Because Qwen can tokenize the same word differently depending on its surrounding space, re-run validation using the same space-prefixed form used during scoring.

In [36]:
from collections import Counter

validation_rows_fixed = []

for item in structural_eval_set:

    source_a_ids = tok.encode(
        item["source_a"],
        add_special_tokens=False
    )

    source_b_ids = tok.encode(
        item["source_b"],
        add_special_tokens=False
    )

    # IMPORTANT: score candidates in their in-context form
    c1_ids = tok.encode(
        " " + item["c1"],
        add_special_tokens=False
    )

    c2_ids = tok.encode(
        " " + item["c2"],
        add_special_tokens=False
    )

    same_source_length = (
        len(source_a_ids) == len(source_b_ids)
    )

    same_candidate_length = (
        len(c1_ids) == len(c2_ids)
    )

    same_candidate_multiset = (
        Counter(c1_ids) == Counter(c2_ids)
    )

    validation_rows_fixed.append({
        "pair_id": item["pair_id"],
        "category": item["category"],
        "source_len_a": len(source_a_ids),
        "source_len_b": len(source_b_ids),
        "candidate_len_c1": len(c1_ids),
        "candidate_len_c2": len(c2_ids),
        "same_source_length": same_source_length,
        "same_candidate_length": same_candidate_length,
        "same_candidate_multiset": same_candidate_multiset,
    })

for row in validation_rows_fixed:
    print(
        f"Pair {row['pair_id']:>2} | "
        f"{row['category']:<24} | "
        f"source={row['source_len_a']}/{row['source_len_b']} | "
        f"cand={row['candidate_len_c1']}/{row['candidate_len_c2']} | "
        f"same_source={row['same_source_length']} | "
        f"same_cand_len={row['same_candidate_length']} | "
        f"same_multiset={row['same_candidate_multiset']}"
    )

Pair  2 | role_argument_binding    | source=17/17 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=True
Pair  3 | modifier_attachment      | source=19/19 | cand=4/4 | same_source=True | same_cand_len=True | same_multiset=True
Pair  4 | spatial_relation         | source=19/19 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=True
Pair  5 | comparative_relation     | source=18/18 | cand=6/6 | same_source=True | same_cand_len=True | same_multiset=True
Pair  6 | role_argument_binding    | source=15/15 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=True
Pair  7 | modifier_attachment      | source=19/19 | cand=4/4 | same_source=True | same_cand_len=True | same_multiset=True
Pair  8 | modifier_attachment      | source=19/19 | cand=4/4 | same_source=True | same_cand_len=True | same_multiset=True
Pair  9 | spatial_relation         | source=20/20 | cand=3/3 | same_source=True | same_cand_len=True | same_multiset=True
Pair 10 | spatial_relati

## Frozen M evaluation

Run the activation-side structural alignment metric on all 14 held-out pairs without changing the Pair 1 settings.

In [37]:
heldout_m_results = []

for item in structural_eval_set:

    pair_layer_results = []

    for layer in sweep_layers:

        # --------------------------------
        # Candidate concept vectors
        # --------------------------------
        concept_vecs = {}

        for label, concept in [
            ("c1", item["c1"]),
            ("c2", item["c2"]),
        ]:

            carrier = (
                carrier_prefix
                + " "
                + concept
                + carrier_suffix
            )

            ids = jlens.encode(
                model,
                tok,
                carrier
            )

            with jlens.record_residuals(
                model,
                [layer]
            ) as rec:

                model.model(
                    input_ids=ids,
                    use_cache=False
                )

            h = rec.acts[layer][0].float().cpu()

            post_h = h[
                post_start :
                post_start + post_window
            ]

            concept_vecs[label] = post_h.mean(dim=0)

        # --------------------------------
        # Source vectors
        # --------------------------------
        source_vecs = {}

        for label, source in [
            ("a", item["source_a"]),
            ("b", item["source_b"]),
        ]:

            ids = jlens.encode(
                model,
                tok,
                source
            )

            with jlens.record_residuals(
                model,
                [layer]
            ) as rec:

                model.model(
                    input_ids=ids,
                    use_cache=False
                )

            h = rec.acts[layer][0].float().cpu()

            source_vecs[label] = h[-1]

        # --------------------------------
        # Structural difference alignment
        # --------------------------------
        source_diff = (
            source_vecs["a"]
            - source_vecs["b"]
        )

        concept_diff = (
            concept_vecs["c1"]
            - concept_vecs["c2"]
        )

        alignment = cos(
            source_diff,
            concept_diff
        )

        pair_layer_results.append(alignment)

    arr = np.array(pair_layer_results)

    result = {
        "pair_id": item["pair_id"],
        "category": item["category"],
        "mean_alignment": arr.mean(),
        "median_alignment": np.median(arr),
        "positive_layers": int((arr > 0).sum()),
        "positive_fraction": (arr > 0).mean(),
        "layer_alignments": pair_layer_results,
    }

    heldout_m_results.append(result)

    print(
        f"Pair {item['pair_id']:>2} | "
        f"{item['category']:<24} | "
        f"mean={result['mean_alignment']:+.4f} | "
        f"median={result['median_alignment']:+.4f} | "
        f"positive={result['positive_layers']}/"
        f"{len(sweep_layers)}"
    )

Pair  2 | role_argument_binding    | mean=+0.1845 | median=+0.1887 | positive=7/7
Pair  3 | modifier_attachment      | mean=+0.2034 | median=+0.1917 | positive=7/7
Pair  4 | spatial_relation         | mean=+0.2523 | median=+0.2815 | positive=7/7
Pair  5 | comparative_relation     | mean=-0.0757 | median=-0.0884 | positive=0/7
Pair  6 | role_argument_binding    | mean=+0.0533 | median=+0.0475 | positive=7/7
Pair  7 | modifier_attachment      | mean=+0.0234 | median=+0.0207 | positive=4/7
Pair  8 | modifier_attachment      | mean=+0.0172 | median=+0.0144 | positive=5/7
Pair  9 | spatial_relation         | mean=+0.0534 | median=+0.0320 | positive=5/7
Pair 10 | spatial_relation         | mean=+0.0216 | median=+0.0132 | positive=5/7
Pair 11 | comparative_relation     | mean=-0.0128 | median=-0.0336 | positive=2/7
Pair 12 | comparative_relation     | mean=-0.1472 | median=-0.1469 | positive=0/7
Pair 13 | quantifier_binding       | mean=-0.0016 | median=+0.0106 | positive=5/7
Pair 14 | quanti

## M results by category

Summarize held-out activation-side alignment across the five relation types.

In [38]:
import pandas as pd

m_df = pd.DataFrame(heldout_m_results)

category_m_summary = (
    m_df
    .groupby("category")
    .agg(
        n_pairs=("pair_id", "count"),
        mean_alignment=("mean_alignment", "mean"),
        median_alignment=("mean_alignment", "median"),
        positive_pairs=("mean_alignment", lambda x: int((x > 0).sum())),
        mean_positive_fraction=("positive_fraction", "mean"),
    )
    .reset_index()
)

category_m_summary["pair_success_rate"] = (
    category_m_summary["positive_pairs"]
    / category_m_summary["n_pairs"]
)

display(category_m_summary)

,category,n_pairs,mean_alignment,median_alignment,positive_pairs,mean_positive_fraction,pair_success_rate
0,comparative_relation,3,-0.078610,-0.075746,0,0.095238,0.000000
1,modifier_attachment,3,0.081357,0.023439,3,0.761905,1.000000
2,quantifier_binding,3,0.004922,-0.001583,1,0.666667,0.333333
3,role_argument_binding,2,0.118906,0.118906,2,1.000000,1.000000
4,spatial_relation,3,0.109081,0.053372,3,0.809524,1.000000


## Frozen L1 evaluation

Run the first-token-conditioned continuation baseline on the same 14 held-out pairs.

A positive paired shift means the source reversal changes continuation preference in the expected direction.

In [39]:
def split_head_tail(candidate):
    ids = tok.encode(
        " " + candidate,
        add_special_tokens=False
    )

    head_id = ids[0]
    tail_ids = ids[1:]

    head = tok.decode([head_id])
    tail = tok.decode(tail_ids)

    return head, tail


heldout_l1_results = []

for item in structural_eval_set:

    c1_head, c1_tail = split_head_tail(item["c1"])
    c2_head, c2_tail = split_head_tail(item["c2"])

    # Source A
    a_c1 = source_head_tail_logprob(
        item["source_a"],
        c1_head,
        c1_tail
    )

    a_c2 = source_head_tail_logprob(
        item["source_a"],
        c2_head,
        c2_tail
    )

    # Source B
    b_c1 = source_head_tail_logprob(
        item["source_b"],
        c1_head,
        c1_tail
    )

    b_c2 = source_head_tail_logprob(
        item["source_b"],
        c2_head,
        c2_tail
    )

    log_odds_a = a_c1 - a_c2
    log_odds_b = b_c1 - b_c2

    paired_shift = (
        log_odds_a
        - log_odds_b
    )

    result = {
        "pair_id": item["pair_id"],
        "category": item["category"],
        "log_odds_a": log_odds_a,
        "log_odds_b": log_odds_b,
        "paired_shift": paired_shift,
        "success": paired_shift > 0,
    }

    heldout_l1_results.append(result)

    print(
        f"Pair {item['pair_id']:>2} | "
        f"{item['category']:<24} | "
        f"L1 shift={paired_shift:+.4f} | "
        f"success={paired_shift > 0}"
    )

Pair  2 | role_argument_binding    | L1 shift=+1.3340 | success=True
Pair  3 | modifier_attachment      | L1 shift=+3.8337 | success=True
Pair  4 | spatial_relation         | L1 shift=+3.1287 | success=True
Pair  5 | comparative_relation     | L1 shift=+1.3490 | success=True
Pair  6 | role_argument_binding    | L1 shift=+1.3855 | success=True
Pair  7 | modifier_attachment      | L1 shift=+2.6502 | success=True
Pair  8 | modifier_attachment      | L1 shift=+2.7116 | success=True
Pair  9 | spatial_relation         | L1 shift=+1.8330 | success=True
Pair 10 | spatial_relation         | L1 shift=+5.6340 | success=True
Pair 11 | comparative_relation     | L1 shift=+1.8109 | success=True
Pair 12 | comparative_relation     | L1 shift=+2.4820 | success=True
Pair 13 | quantifier_binding       | L1 shift=+0.8177 | success=True
Pair 14 | quantifier_binding       | L1 shift=+0.7953 | success=True
Pair 15 | quantifier_binding       | L1 shift=+0.5388 | success=True


## Common directional comparison

M and L1 use different score spaces, so their raw values are not directly compared.

Instead compare whether each method moves in the expected direction for each held-out pair.

In [41]:
m_compare = pd.DataFrame([
    {
        "pair_id": r["pair_id"],
        "category": r["category"],
        "M_score": r["mean_alignment"],
        "M_success": r["mean_alignment"] > 0,
    }
    for r in heldout_m_results
])

l1_compare = pd.DataFrame([
    {
        "pair_id": r["pair_id"],
        "L1_score": r["paired_shift"],
        "L1_success": r["success"],
    }
    for r in heldout_l1_results
])

comparison_df = m_compare.merge(
    l1_compare,
    on="pair_id"
)

comparison_df["comparison"] = comparison_df.apply(
    lambda r:
        "both"
        if r["M_success"] and r["L1_success"]
        else "L1_only"
        if (not r["M_success"]) and r["L1_success"]
        else "M_only"
        if r["M_success"] and (not r["L1_success"])
        else "neither",
    axis=1
)

display(
    comparison_df[
        [
            "pair_id",
            "category",
            "M_success",
            "L1_success",
            "comparison",
        ]
    ]
)

print()
print(
    "M success:",
    int(comparison_df["M_success"].sum()),
    "/",
    len(comparison_df)
)

print(
    "L1 success:",
    int(comparison_df["L1_success"].sum()),
    "/",
    len(comparison_df)
)

print()
print(comparison_df["comparison"].value_counts())

,pair_id,category,M_success,L1_success,comparison
0,2,role_argument_binding,True,True,both
1,3,modifier_attachment,True,True,both
2,4,spatial_relation,True,True,both
3,5,comparative_relation,False,True,L1_only
4,6,role_argument_binding,True,True,both
5,7,modifier_attachment,True,True,both
6,8,modifier_attachment,True,True,both
7,9,spatial_relation,True,True,both
8,10,spatial_relation,True,True,both
9,11,comparative_relation,False,True,L1_only



M success: 9 / 14
L1 success: 14 / 14

comparison
both       9
L1_only    5
Name: count, dtype: int64


## L1 mismatched-source control

Break the correct source-candidate correspondence and check whether the L1 signal weakens.

In [42]:
l1_mismatch_results = []

n = len(structural_eval_set)

for i, item in enumerate(structural_eval_set):

    # Use sources from the NEXT pair instead of the correct pair
    wrong_source_item = structural_eval_set[(i + 1) % n]

    c1_head, c1_tail = split_head_tail(item["c1"])
    c2_head, c2_tail = split_head_tail(item["c2"])

    # Wrong Source A
    a_c1 = source_head_tail_logprob(
        wrong_source_item["source_a"],
        c1_head,
        c1_tail
    )

    a_c2 = source_head_tail_logprob(
        wrong_source_item["source_a"],
        c2_head,
        c2_tail
    )

    # Wrong Source B
    b_c1 = source_head_tail_logprob(
        wrong_source_item["source_b"],
        c1_head,
        c1_tail
    )

    b_c2 = source_head_tail_logprob(
        wrong_source_item["source_b"],
        c2_head,
        c2_tail
    )

    log_odds_a = a_c1 - a_c2
    log_odds_b = b_c1 - b_c2

    paired_shift = log_odds_a - log_odds_b

    l1_mismatch_results.append({
        "pair_id": item["pair_id"],
        "wrong_source_pair": wrong_source_item["pair_id"],
        "paired_shift": paired_shift,
        "success": paired_shift > 0,
    })

    print(
        f"Candidate Pair {item['pair_id']:>2} "
        f"<- Source Pair {wrong_source_item['pair_id']:>2} | "
        f"shift={paired_shift:+.4f} | "
        f"success={paired_shift > 0}"
    )

print()
print(
    "Mismatched-source L1 success:",
    sum(r["success"] for r in l1_mismatch_results),
    "/",
    len(l1_mismatch_results)
)

Candidate Pair  2 <- Source Pair  3 | shift=+0.0293 | success=True
Candidate Pair  3 <- Source Pair  4 | shift=-0.0104 | success=False
Candidate Pair  4 <- Source Pair  5 | shift=-0.0195 | success=False
Candidate Pair  5 <- Source Pair  6 | shift=+0.0449 | success=True
Candidate Pair  6 <- Source Pair  7 | shift=+0.1367 | success=True
Candidate Pair  7 <- Source Pair  8 | shift=+0.1250 | success=True
Candidate Pair  8 <- Source Pair  9 | shift=-0.1094 | success=False
Candidate Pair  9 <- Source Pair 10 | shift=+0.0625 | success=True
Candidate Pair 10 <- Source Pair 11 | shift=+0.2344 | success=True
Candidate Pair 11 <- Source Pair 12 | shift=+0.0016 | success=True
Candidate Pair 12 <- Source Pair 13 | shift=+0.0039 | success=True
Candidate Pair 13 <- Source Pair 14 | shift=+0.2119 | success=True
Candidate Pair 14 <- Source Pair 15 | shift=+0.2275 | success=True
Candidate Pair 15 <- Source Pair  2 | shift=-0.0859 | success=False

Mismatched-source L1 success: 10 / 14


## Matched versus mismatched L1 magnitude

Compare the magnitude of the correct L1 structural shifts with shifts produced by mismatched sources.

In [43]:
matched_shifts = np.array([
    r["paired_shift"]
    for r in heldout_l1_results
])

mismatched_shifts = np.array([
    r["paired_shift"]
    for r in l1_mismatch_results
])

print("MATCHED")
print("Mean shift:       ", matched_shifts.mean())
print("Median shift:     ", np.median(matched_shifts))
print("Mean abs shift:   ", np.abs(matched_shifts).mean())

print()

print("MISMATCHED")
print("Mean shift:       ", mismatched_shifts.mean())
print("Median shift:     ", np.median(mismatched_shifts))
print("Mean abs shift:   ", np.abs(mismatched_shifts).mean())

print()

print(
    "Matched > mismatched magnitude:",
    int(
        (
            np.abs(matched_shifts)
            > np.abs(mismatched_shifts)
        ).sum()
    ),
    "/",
    len(matched_shifts)
)

print(
    "Mean magnitude ratio:",
    np.abs(matched_shifts).mean()
    / np.abs(mismatched_shifts).mean()
)

MATCHED
Mean shift:        2.1645989554268974
Median shift:      1.8219512939453124
Mean abs shift:    2.1645989554268974

MISMATCHED
Mean shift:        0.060890997023809526
Median shift:      0.03710937499999978
Mean abs shift:    0.09307105654761896

Matched > mismatched magnitude: 14 / 14
Mean magnitude ratio: 23.257487727340884


## L1 permutation null

Repeat the source mismatch across 20 random permutations to estimate how large the L1 signal is without the correct source-candidate correspondence.

In [44]:
import random

random.seed(42)

n_permutations = 20
permutation_mean_abs_shifts = []

n = len(structural_eval_set)

for perm_idx in range(n_permutations):

    # Create a derangement:
    # no candidate pair receives its own source pair
    while True:
        perm = list(range(n))
        random.shuffle(perm)

        if all(perm[i] != i for i in range(n)):
            break

    perm_shifts = []

    for i, item in enumerate(structural_eval_set):

        wrong_source_item = structural_eval_set[perm[i]]

        c1_head, c1_tail = split_head_tail(item["c1"])
        c2_head, c2_tail = split_head_tail(item["c2"])

        a_c1 = source_head_tail_logprob(
            wrong_source_item["source_a"],
            c1_head,
            c1_tail
        )

        a_c2 = source_head_tail_logprob(
            wrong_source_item["source_a"],
            c2_head,
            c2_tail
        )

        b_c1 = source_head_tail_logprob(
            wrong_source_item["source_b"],
            c1_head,
            c1_tail
        )

        b_c2 = source_head_tail_logprob(
            wrong_source_item["source_b"],
            c2_head,
            c2_tail
        )

        shift = (
            (a_c1 - a_c2)
            - (b_c1 - b_c2)
        )

        perm_shifts.append(shift)

    mean_abs = np.abs(
        np.array(perm_shifts)
    ).mean()

    permutation_mean_abs_shifts.append(mean_abs)

print("Matched mean abs shift:")
print(np.abs(matched_shifts).mean())

print()

print("Permutation null mean abs shift:")
print(np.mean(permutation_mean_abs_shifts))

print("Permutation null median:")
print(np.median(permutation_mean_abs_shifts))

print("Permutation null max:")
print(np.max(permutation_mean_abs_shifts))

print()

print(
    "Matched / null-mean ratio:",
    np.abs(matched_shifts).mean()
    / np.mean(permutation_mean_abs_shifts)
)

Matched mean abs shift:
2.1645989554268974

Permutation null mean abs shift:
0.1370492408389137
Permutation null median:
0.11686023530505953
Permutation null max:
0.23643043154761895

Matched / null-mean ratio: 15.794315548023686


## M pair-permutation control

Break the correct correspondence between source-difference and concept-difference vectors while keeping the frozen M metric unchanged.

In [45]:
# Cache source and concept difference vectors
# for every held-out pair and every frozen layer.

m_diff_cache = {}

for item in structural_eval_set:

    pair_id = item["pair_id"]

    # -----------------------------------
    # Candidate concept representations
    # -----------------------------------
    concept_by_label = {}

    for label, concept in [
        ("c1", item["c1"]),
        ("c2", item["c2"]),
    ]:

        carrier = (
            carrier_prefix
            + " "
            + concept
            + carrier_suffix
        )

        ids = jlens.encode(model, tok, carrier)

        with jlens.record_residuals(
            model,
            sweep_layers
        ) as rec:

            model.model(
                input_ids=ids,
                use_cache=False
            )

        concept_by_label[label] = {}

        for layer in sweep_layers:

            h = rec.acts[layer][0].float().cpu()

            post_h = h[
                post_start :
                post_start + post_window
            ]

            concept_by_label[label][layer] = (
                post_h.mean(dim=0)
            )

    # -----------------------------------
    # Source representations
    # -----------------------------------
    source_by_label = {}

    for label, source in [
        ("a", item["source_a"]),
        ("b", item["source_b"]),
    ]:

        ids = jlens.encode(model, tok, source)

        with jlens.record_residuals(
            model,
            sweep_layers
        ) as rec:

            model.model(
                input_ids=ids,
                use_cache=False
            )

        source_by_label[label] = {}

        for layer in sweep_layers:

            h = rec.acts[layer][0].float().cpu()

            source_by_label[label][layer] = h[-1]

    # -----------------------------------
    # Cache difference directions
    # -----------------------------------
    m_diff_cache[pair_id] = {}

    for layer in sweep_layers:

        m_diff_cache[pair_id][layer] = {
            "source_diff": (
                source_by_label["a"][layer]
                - source_by_label["b"][layer]
            ),
            "concept_diff": (
                concept_by_label["c1"][layer]
                - concept_by_label["c2"][layer]
            ),
        }

print("Cached pairs:", len(m_diff_cache))

Cached pairs: 14


## M permutation null

Use 500 Monte Carlo pair permutations to build a null distribution for the matched mean structural alignment.

In [46]:
random.seed(42)

pair_ids = [item["pair_id"] for item in structural_eval_set]

# ----------------------------------------
# Correctly matched M statistics
# ----------------------------------------
matched_pair_means = np.array([
    r["mean_alignment"]
    for r in heldout_m_results
])

matched_global_mean = matched_pair_means.mean()
matched_positive_pairs = int((matched_pair_means > 0).sum())

print("MATCHED M")
print("Global mean alignment:", matched_global_mean)
print(
    "Positive pairs:",
    matched_positive_pairs,
    "/",
    len(pair_ids)
)

# ----------------------------------------
# Permutation null
# ----------------------------------------
n_permutations = 500

null_global_means = []
null_positive_counts = []

for _ in range(n_permutations):

    # Derangement: no pair gets its own concept vector
    while True:
        shuffled_ids = pair_ids.copy()
        random.shuffle(shuffled_ids)

        if all(
            shuffled_ids[i] != pair_ids[i]
            for i in range(len(pair_ids))
        ):
            break

    perm_pair_means = []

    for source_pair, concept_pair in zip(
        pair_ids,
        shuffled_ids
    ):

        layer_alignments = []

        for layer in sweep_layers:

            source_diff = (
                m_diff_cache[source_pair][layer]["source_diff"]
            )

            concept_diff = (
                m_diff_cache[concept_pair][layer]["concept_diff"]
            )

            layer_alignments.append(
                cos(source_diff, concept_diff)
            )

        perm_pair_means.append(
            np.mean(layer_alignments)
        )

    perm_pair_means = np.array(perm_pair_means)

    null_global_means.append(
        perm_pair_means.mean()
    )

    null_positive_counts.append(
        int((perm_pair_means > 0).sum())
    )

null_global_means = np.array(null_global_means)
null_positive_counts = np.array(null_positive_counts)

print()
print("PERMUTATION NULL")
print(
    "Mean global alignment:",
    null_global_means.mean()
)
print(
    "95th percentile:",
    np.percentile(null_global_means, 95)
)
print(
    "Max global alignment:",
    null_global_means.max()
)

print()
print(
    "Mean positive-pair count:",
    null_positive_counts.mean()
)
print(
    "95th percentile positive count:",
    np.percentile(null_positive_counts, 95)
)

# Empirical one-sided p-values
p_mean = (
    1
    + np.sum(null_global_means >= matched_global_mean)
) / (n_permutations + 1)

p_positive = (
    1
    + np.sum(null_positive_counts >= matched_positive_pairs)
) / (n_permutations + 1)

print()
print("Empirical p (mean alignment):", p_mean)
print("Empirical p (positive pairs):", p_positive)

MATCHED M
Global mean alignment: 0.04200427130230571
Positive pairs: 9 / 14

PERMUTATION NULL
Mean global alignment: -0.0020694622421870007
95th percentile: 0.017852476586549686
Max global alignment: 0.03935324800039204

Mean positive-pair count: 6.576
95th percentile positive count: 9.049999999999955

Empirical p (mean alignment): 0.001996007984031936
Empirical p (positive pairs): 0.1497005988023952


## Core experiment summary

The frozen evaluation used 14 held-out pairs.

### L1

- expected direction: 14/14
- matched mean absolute shift: 2.1646
- permutation-null mean absolute shift: 0.1370

### M

- positive pairs: 9/14
- matched mean alignment: +0.0420
- permutation-null mean: -0.0021
- empirical one-sided p: about 0.002

The 9/14 positive count alone was not significant, with p about 0.15.

### Directional comparison

- both: 9
- L1 only: 5
- M only: 0
- neither: 0

Both signals were present, but continuation was much more consistent.

# Gate 3: PASS

The frozen held-out experiment is complete.

M shows pair-specific activation-side structural signal beyond its permutation null, while L1 tracks the tested structural reversals more consistently.

This result applies to the lightweight M metric used here and is not a negative result about full multi-token J-Lens.

In [47]:
import pandas as pd
import json
from pathlib import Path

results_dir = Path("/workspace/multi-token-jlens/results")
results_dir.mkdir(exist_ok=True)

# Save M results
pd.DataFrame(heldout_m_results).to_json(
    results_dir / "heldout_m_results.json",
    orient="records",
    indent=2,
)

# Save L1 results
pd.DataFrame(heldout_l1_results).to_json(
    results_dir / "heldout_l1_results.json",
    orient="records",
    indent=2,
)

print("Saved:")
print(results_dir / "heldout_m_results.json")
print(results_dir / "heldout_l1_results.json")

Saved:
/workspace/multi-token-jlens/results/heldout_m_results.json
/workspace/multi-token-jlens/results/heldout_l1_results.json


# Post-Gate checks

The analyses below were added after the frozen Gate 3 result and are treated as exploratory robustness checks.

## M+ robustness check

The comparative-relation pairs were all negative under M.

As a post-hoc check, estimate residual variance from 100 WikiText prompts and apply diagonal whitening to reduce the influence of unusually high-variance dimensions.

This does not change the frozen Gate 3 result.

In [48]:
# 3.36 — Inspect existing objects before M+ calibration

names_to_check = [
    "model",
    "tokenizer",
    "heldout_m_results",
    "layers",
    "carrier_prompt",
    "carrier_layer",
    "candidate_vectors",
    "source_diff_vectors",
    "concept_diff_vectors",
]

for name in names_to_check:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: FOUND | type={type(obj).__name__}")
    else:
        print(f"{name}: NOT FOUND")

model: FOUND | type=Qwen3_5ForCausalLM
tokenizer: NOT FOUND
heldout_m_results: FOUND | type=list
layers: FOUND | type=list
carrier_prompt: NOT FOUND
carrier_layer: FOUND | type=int
candidate_vectors: NOT FOUND
source_diff_vectors: NOT FOUND
concept_diff_vectors: NOT FOUND


In [49]:
# 3.36b — Find relevant existing variable names

keywords = [
    "token",
    "carrier",
    "prompt",
    "source",
    "concept",
    "candidate",
    "heldout",
    "activation",
    "vector",
    "wiki",
]

for kw in keywords:
    matches = [
        name for name in globals().keys()
        if kw.lower() in name.lower() and not name.startswith("_")
    ]
    print(f"\n[{kw}]")
    print(matches[:50])


[token]
['target_tokens', 'first_target_token', 'top5_tokens', 'tokens', 'top_tokens', 'token_ids']

[carrier]
['carrier_template', 'carrier', 'carrier_prefix', 'carrier_suffix', 'carrier_ids', 'carrier_layer']

[prompt]
['prompt']

[source]
['source_ids', 'neutral_source', 'source_a', 'source_b', 'source_head_tail_logprob', 'source_hidden_states', 'source_h', 'source_a_h', 'source_b_h', 'source_difference', 'source_vecs_layer', 'source_diff', 'source_a_ids', 'source_b_ids', 'same_source_length', 'source_vecs', 'source', 'wrong_source_item', 'source_by_label', 'source_pair']

[concept]
['concept', 'concept_ids', 'concept_ids_in_context', 'concept_hidden_states', 'concept_vectors', 'concept_difference', 'concept_vecs_layer', 'concept_diff', 'concept_vecs', 'concept_by_label', 'concept_pair']

[candidate]
['candidate_logprob', 'cosine_between_candidates', 'same_candidate_length', 'same_candidate_multiset']

[heldout]
['heldout_m_results', 'heldout_l1_results']

[activation]
[]

[vector]

In [50]:
# 3.36c — Inspect shapes/types of existing M-related objects

objects_to_inspect = [
    "carrier",
    "carrier_ids",
    "source_a_h",
    "source_b_h",
    "source_diff",
    "source_vecs",
    "concept_vectors",
    "concept_difference",
    "concept_vecs_layer",
]

for name in objects_to_inspect:
    if name in globals():
        obj = globals()[name]

        shape = getattr(obj, "shape", None)

        if shape is not None:
            print(f"{name}: type={type(obj).__name__}, shape={shape}")
        elif isinstance(obj, (list, tuple)):
            print(f"{name}: type={type(obj).__name__}, len={len(obj)}")
            if len(obj) > 0:
                first = obj[0]
                print(
                    f"    first item: type={type(first).__name__}, "
                    f"shape={getattr(first, 'shape', None)}"
                )
        elif isinstance(obj, dict):
            print(f"{name}: type=dict, keys={list(obj.keys())[:10]}")
        else:
            print(f"{name}: type={type(obj).__name__}")
    else:
        print(f"{name}: NOT FOUND")

carrier: type=str
carrier_ids: type=list, len=24
    first item: type=int, shape=None
source_a_h: type=Tensor, shape=torch.Size([2560])
source_b_h: type=Tensor, shape=torch.Size([2560])
source_diff: type=Tensor, shape=torch.Size([2560])
source_vecs: type=dict, keys=['a', 'b']
concept_vectors: type=dict, keys=['user tests model', 'model tests user']
concept_difference: type=Tensor, shape=torch.Size([2560])
concept_vecs_layer: type=dict, keys=['user tests model', 'model tests user']


In [51]:
# 3.36d — Inspect vector contents and locate tokenizer object

print("SOURCE VECS")
print("keys:", source_vecs.keys())
for k, v in source_vecs.items():
    print(k, type(v).__name__, getattr(v, "shape", None))

print("\nCONCEPT VECS BY LAYER")
print("keys:", concept_vecs_layer.keys())
for concept_name, layer_dict in concept_vecs_layer.items():
    print(f"\n{concept_name}: type={type(layer_dict).__name__}")
    if isinstance(layer_dict, dict):
        for layer_key, vec in list(layer_dict.items())[:10]:
            print(
                f"  layer={layer_key} | "
                f"type={type(vec).__name__} | "
                f"shape={getattr(vec, 'shape', None)}"
            )
    else:
        print("  shape:", getattr(layer_dict, "shape", None))

print("\nTOKENIZER-LIKE OBJECTS")
for name, obj in globals().items():
    type_name = type(obj).__name__.lower()
    if "tokenizer" in type_name:
        print(name, "->", type(obj).__name__)

SOURCE VECS
keys: dict_keys(['a', 'b'])
a Tensor torch.Size([2560])
b Tensor torch.Size([2560])

CONCEPT VECS BY LAYER
keys: dict_keys(['user tests model', 'model tests user'])

user tests model: type=Tensor
  shape: torch.Size([2560])

model tests user: type=Tensor
  shape: torch.Size([2560])

TOKENIZER-LIKE OBJECTS


RuntimeError: dictionary changed size during iteration

In [52]:
# 3.36e — Safely locate tokenizer-like objects

for name, obj in list(globals().items()):
    type_name = type(obj).__name__.lower()
    if "tokenizer" in type_name:
        print(name, "->", type(obj).__name__)

tok -> Qwen2Tokenizer


In [53]:
# 3.36f — Inspect reusable helper functions for calibration activations

keywords = [
    "hidden",
    "resid",
    "activation",
    "source",
    "concept",
    "layer",
    "prompt",
]

for kw in keywords:
    matches = []
    for name, obj in list(globals().items()):
        if kw.lower() in name.lower() and callable(obj):
            matches.append(name)

    print(f"\n[{kw}]")
    print(matches[:50])


[hidden]
[]

[resid]
[]

[activation]
[]

[source]
['source_head_tail_logprob']

[concept]
[]

[layer]
[]

[prompt]
[]


In [54]:
# 3.36g — Look for cached calibration / WikiText prompt collections

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    # Lists/tuples that might contain prompt strings
    if isinstance(obj, (list, tuple)):
        if 50 <= len(obj) <= 200:
            sample = obj[0] if len(obj) > 0 else None
            print(
                f"{name}: type={type(obj).__name__}, len={len(obj)}, "
                f"first_type={type(sample).__name__}"
            )

    # Dataset-like / text-like objects
    type_name = type(obj).__name__.lower()
    if "dataset" in type_name:
        try:
            print(f"{name}: dataset-like, len={len(obj)}")
        except:
            print(f"{name}: dataset-like")

In: type=list, len=55, first_type=str


In [55]:
# 3.36h — Find how WikiText calibration data was created in Phase 1

from pathlib import Path
import json

nb_path = Path("/workspace/multi-token-jlens/notebooks/01_reproduce_jlens.ipynb")

nb = json.loads(nb_path.read_text())

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if any(
        kw in source.lower()
        for kw in ["wikitext", "wiki", "dataset", "calibration"]
    ):
        print(f"\n--- CELL {i} ---")
        print(source[:4000])


--- CELL 4 ---
import sys
import importlib.util

print("Python:", sys.executable)
print("transformers:", importlib.util.find_spec("transformers"))
print("datasets:", importlib.util.find_spec("datasets"))
print("huggingface_hub:", importlib.util.find_spec("huggingface_hub"))

--- CELL 11 ---
N_FIT_PROMPTS = 100

fit_prompts = jlens.load_wikitext(N_FIT_PROMPTS)

print("Number of fit prompts:", len(fit_prompts))
print("\nFirst prompt preview:")
print(fit_prompts[0][:500])


In [56]:
# 3.36i — Check Gate 1 calibration loader availability

print("jlens:", "FOUND" if "jlens" in globals() else "NOT FOUND")
print("N_FIT_PROMPTS:", globals().get("N_FIT_PROMPTS", "NOT FOUND"))

jlens: FOUND
N_FIT_PROMPTS: NOT FOUND


In [57]:
# 3.36j — Reload the same WikiText calibration prompts used in Gate 1

N_FIT_PROMPTS = 100
fit_prompts = jlens.load_wikitext(N_FIT_PROMPTS)

print("Number of calibration prompts:", len(fit_prompts))
print("\nFirst prompt preview:")
print(fit_prompts[0][:500])

Number of calibration prompts: 100

First prompt preview:
 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and f


In [58]:
# 3.36k — Sanity-check hidden-state extraction on one calibration prompt

import torch

test_prompt = fit_prompts[0]

inputs = tok(
    test_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False,
    )

print("num hidden-state tensors:", len(out.hidden_states))
print("input length:", inputs["input_ids"].shape[1])

for idx in [0, 20, 24, 28, 32]:
    if idx < len(out.hidden_states):
        print(
            f"hidden_states[{idx}] shape:",
            out.hidden_states[idx].shape
        )

num hidden-state tensors: 33
input length: 166
hidden_states[0] shape: torch.Size([1, 166, 2560])
hidden_states[20] shape: torch.Size([1, 166, 2560])
hidden_states[24] shape: torch.Size([1, 166, 2560])
hidden_states[28] shape: torch.Size([1, 166, 2560])
hidden_states[32] shape: torch.Size([1, 166, 2560])


In [59]:
# 3.36l — Verify the exact hidden-state/layer indexing used in the original M experiment

from pathlib import Path
import json

nb_path = Path("/workspace/multi-token-jlens/notebooks/03_core_experiment.ipynb")
nb = json.loads(nb_path.read_text())

search_terms = [
    "source_hidden_states",
    "concept_hidden_states",
    "source_vecs_layer",
    "concept_vecs_layer",
    "carrier_layer",
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if any(term in source for term in search_terms):
        print(f"\n--- CELL {i} ---")
        print(source[:5000])


--- CELL 43 ---
carrier_layer = 24
post_start = 14
post_window = 4

concept_hidden_states = {}

for concept in [
    "user tests model",
    "model tests user",
]:
    carrier = carrier_prefix + " " + concept + carrier_suffix

    ids = jlens.encode(model, tok, carrier)

    with jlens.record_residuals(model, [carrier_layer]) as rec:
        model.model(
            input_ids=ids,
            use_cache=False
        )

    h = rec.acts[carrier_layer][0].float().cpu()

    post_h = h[
        post_start : post_start + post_window
    ]

    concept_hidden_states[concept] = post_h

    token_ids = ids[0].tolist()

    print("=" * 70)
    print("Concept:", concept)
    print("Full hidden-state shape:", tuple(h.shape))
    print("Post-concept shape:", tuple(post_h.shape))
    print()

    for pos in range(post_start, post_start + post_window):
        print(
            f"position {pos}:",
            repr(tok.decode([token_ids[pos]]))
        )

    print()

--- CELL 45 ---
import torch


In [60]:
# 3.36n — Build WikiText calibration statistics using the SAME residual convention as M

import torch

calibration_layers = [20, 22, 24, 25, 26, 27, 28]

# For each layer, collect one residual vector per WikiText prompt
# using the final non-padding token, matching the source-side M extraction.
calibration_acts = {layer: [] for layer in calibration_layers}

for i, prompt_text in enumerate(fit_prompts):

    ids = jlens.encode(model, tok, prompt_text)

    # Keep calibration prompts reasonably bounded
    ids = ids[:, :256]

    with torch.no_grad():
        with jlens.record_residuals(model, calibration_layers) as rec:
            model.model(
                input_ids=ids,
                use_cache=False
            )

    for layer in calibration_layers:
        h = rec.acts[layer][0].float().cpu()

        # Final token residual, same extraction convention as source vectors
        calibration_acts[layer].append(h[-1])

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(fit_prompts)} prompts")


calibration_stats = {}

for layer in calibration_layers:

    X = torch.stack(calibration_acts[layer], dim=0)   # [100, 2560]

    mean = X.mean(dim=0)
    std = X.std(dim=0, unbiased=False)

    # Numerical safety only
    std = std.clamp_min(1e-6)

    calibration_stats[layer] = {
        "mean": mean,
        "std": std,
        "n": X.shape[0],
    }

    print(
        f"L{layer}: "
        f"X={tuple(X.shape)} | "
        f"mean_norm={mean.norm().item():.3f} | "
        f"std_mean={std.mean().item():.3f} | "
        f"std_min={std.min().item():.6f}"
    )

Processed 20/100 prompts
Processed 40/100 prompts
Processed 60/100 prompts
Processed 80/100 prompts
Processed 100/100 prompts
L20: X=(100, 2560) | mean_norm=11.047 | std_mean=0.208 | std_min=0.102113
L22: X=(100, 2560) | mean_norm=16.725 | std_mean=0.277 | std_min=0.140735
L24: X=(100, 2560) | mean_norm=17.952 | std_mean=0.330 | std_min=0.165508
L25: X=(100, 2560) | mean_norm=19.565 | std_mean=0.351 | std_min=0.176099
L26: X=(100, 2560) | mean_norm=26.684 | std_mean=0.381 | std_min=0.205208
L27: X=(100, 2560) | mean_norm=21.890 | std_mean=0.421 | std_min=0.237474
L28: X=(100, 2560) | mean_norm=27.656 | std_mean=0.481 | std_min=0.269662


In [61]:
# 3.36o — Shared layer-wise calibration statistics for M+
# Use the SAME mean/std transform for both source and concept vectors.

calibration_layers = [20, 22, 24, 25, 26, 27, 28]

# Streaming accumulators: avoids storing every token activation in memory
sum_x = {
    layer: torch.zeros(2560, dtype=torch.float64)
    for layer in calibration_layers
}
sum_x2 = {
    layer: torch.zeros(2560, dtype=torch.float64)
    for layer in calibration_layers
}
count_x = {
    layer: 0
    for layer in calibration_layers
}

for i, prompt_text in enumerate(fit_prompts):

    ids = jlens.encode(model, tok, prompt_text)
    ids = ids[:, :256]

    with torch.no_grad():
        with jlens.record_residuals(model, calibration_layers) as rec:
            model.model(
                input_ids=ids,
                use_cache=False
            )

    for layer in calibration_layers:
        # [seq_len, 2560]
        h = rec.acts[layer][0].float().cpu().double()

        sum_x[layer] += h.sum(dim=0)
        sum_x2[layer] += (h * h).sum(dim=0)
        count_x[layer] += h.shape[0]

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(fit_prompts)} prompts")


shared_calibration_stats = {}

for layer in calibration_layers:
    n = count_x[layer]

    mean = sum_x[layer] / n
    var = (sum_x2[layer] / n) - mean.pow(2)
    var = var.clamp_min(1e-12)

    std = var.sqrt().clamp_min(1e-6)

    # Back to float32 for later vector transforms
    shared_calibration_stats[layer] = {
        "mean": mean.float(),
        "std": std.float(),
        "n_tokens": n,
    }

    print(
        f"L{layer}: "
        f"n_tokens={n} | "
        f"mean_norm={mean.norm().item():.3f} | "
        f"std_mean={std.mean().item():.3f} | "
        f"std_min={std.min().item():.6f}"
    )

Processed 20/100 prompts
Processed 40/100 prompts
Processed 60/100 prompts
Processed 80/100 prompts
Processed 100/100 prompts
L20: n_tokens=21013 | mean_norm=9.023 | std_mean=0.276 | std_min=0.172927
L22: n_tokens=21013 | mean_norm=12.754 | std_mean=0.374 | std_min=0.238630
L24: n_tokens=21013 | mean_norm=14.036 | std_mean=0.464 | std_min=0.316862
L25: n_tokens=21013 | mean_norm=15.633 | std_mean=0.506 | std_min=0.361701
L26: n_tokens=21013 | mean_norm=19.493 | std_mean=0.551 | std_min=0.395235
L27: n_tokens=21013 | mean_norm=19.309 | std_mean=0.599 | std_min=0.433929
L28: n_tokens=21013 | mean_norm=23.952 | std_mean=0.676 | std_min=0.494478


In [62]:
# 3.37 — Sanity-check centering and diagonal whitening on Pair 1

layer = 24

mu = shared_calibration_stats[layer]["mean"]
sigma = shared_calibration_stats[layer]["std"]

# Original Pair 1 vectors from the frozen M experiment
s_a = source_a_h.float().cpu()
s_b = source_b_h.float().cpu()

c1 = concept_vectors["user tests model"].float().cpu()
c2 = concept_vectors["model tests user"].float().cpu()


# ---------------------------------------
# 1. Original difference directions
# ---------------------------------------
raw_source_diff = s_a - s_b
raw_concept_diff = c1 - c2


# ---------------------------------------
# 2. Center each vector, then difference
#    (mean should cancel exactly)
# ---------------------------------------
centered_source_diff = (s_a - mu) - (s_b - mu)
centered_concept_diff = (c1 - mu) - (c2 - mu)


# ---------------------------------------
# 3. Diagonal whitening
# ---------------------------------------
mplus_source_diff = raw_source_diff / sigma
mplus_concept_diff = raw_concept_diff / sigma


raw_alignment = cos(
    raw_source_diff,
    raw_concept_diff
)

centered_alignment = cos(
    centered_source_diff,
    centered_concept_diff
)

mplus_alignment = cos(
    mplus_source_diff,
    mplus_concept_diff
)


print(
    "Max |raw - centered| source:",
    (raw_source_diff - centered_source_diff).abs().max().item()
)

print(
    "Max |raw - centered| concept:",
    (raw_concept_diff - centered_concept_diff).abs().max().item()
)

print()
print("Original M alignment:", raw_alignment)
print("Centered alignment:", centered_alignment)
print("M+ whitened alignment:", mplus_alignment)

Max |raw - centered| source: 1.1920928955078125e-07
Max |raw - centered| concept: 5.960464477539063e-08

Original M alignment: 0.08554947376251221
Centered alignment: 0.08554947376251221
M+ whitened alignment: 0.08729102462530136


In [63]:
# 3.38a — Locate held-out pair definitions for comparative relations

keywords = [
    "heldout",
    "pair",
    "comparative",
    "source_a",
    "source_b",
]

for kw in keywords:
    matches = []
    for name, obj in list(globals().items()):
        if kw.lower() in name.lower():
            matches.append(
                (
                    name,
                    type(obj).__name__,
                    getattr(obj, "shape", None),
                    len(obj) if hasattr(obj, "__len__") else None,
                )
            )

    print(f"\n[{kw}]")
    for item in matches[:50]:
        print(item)


[heldout]
('heldout_m_results', 'list', None, 14)
('heldout_l1_results', 'list', None, 14)

[pair]
('pair1_targets', 'dict', None, 2)
('pair1_targets_spaced', 'dict', None, 2)
('pair1_s', 'dict', None, 2)
('pair1_structural_pilot', 'dict', None, 2)
('paired_shift', 'float', None, None)
('l1_paired_shift', 'float', None, None)
('m_paired_shift', 'float', None, None)
('pair_layer_results', 'list', None, 7)
('pair_id', 'int', None, None)
('pair_ids', 'list', None, 14)
('matched_pair_means', 'ndarray', (14,), 14)
('matched_positive_pairs', 'int', None, None)
('perm_pair_means', 'ndarray', (14,), 14)
('source_pair', 'int', None, None)
('concept_pair', 'int', None, None)

[comparative]

[source_a]
('source_a', 'str', None, 68)
('source_a_h', 'Tensor', torch.Size([2560]), 2560)
('source_a_ids', 'list', None, 13)

[source_b]
('source_b', 'str', None, 68)
('source_b_h', 'Tensor', torch.Size([2560]), 2560)
('source_b_ids', 'list', None, 13)
('source_by_label', 'dict', None, 2)


In [64]:
# 3.38b — Inspect held-out result structure and locate comparative pairs

print("heldout_m_results length:", len(heldout_m_results))
print("\nKeys in first result:")
print(heldout_m_results[0].keys())

print("\nComparative-relation entries:")
for row in heldout_m_results:
    if row.get("category") == "comparative_relation":
        print("\n---")
        for k, v in row.items():
            print(f"{k}: {v}")

heldout_m_results length: 14

Keys in first result:
dict_keys(['pair_id', 'category', 'mean_alignment', 'median_alignment', 'positive_layers', 'positive_fraction', 'layer_alignments'])

Comparative-relation entries:

---
pair_id: 5
category: comparative_relation
mean_alignment: -0.07574626404259886
median_alignment: -0.08835344761610031
positive_layers: 0
positive_fraction: 0.0
layer_alignments: [-0.12570548057556152, -0.12454862892627716, -0.08835344761610031, -0.09359700977802277, -0.02378069795668125, -0.03364094719290733, -0.04059763625264168]

---
pair_id: 11
category: comparative_relation
mean_alignment: -0.012833744287490845
median_alignment: -0.03358267992734909
positive_layers: 2
positive_fraction: 0.2857142857142857
layer_alignments: [-0.13259415328502655, -0.05091559886932373, -0.04443049803376198, -0.03358267992734909, -0.021602969616651535, 0.0863996222615242, 0.10689006745815277]

---
pair_id: 12
category: comparative_relation
mean_alignment: -0.14724927076271602
median_a

In [65]:
# 3.38c — Find the original held-out source/candidate definitions

from pathlib import Path
import json

nb_path = Path("/workspace/multi-token-jlens/notebooks/03_core_experiment.ipynb")
nb = json.loads(nb_path.read_text())

search_terms = [
    "green box heavier",
    "train longer",
    "blue car faster",
    "heldout_structural",
    "heldout_pairs",
    "source_a",
    "source_b",
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    if any(term.lower() in source.lower() for term in search_terms):
        print(f"\n--- CELL {i} ---")
        print(source[:8000])


--- CELL 35 ---
c1 = "user tests model"
c2 = "model tests user"

source_a = pair1_structural_pilot["A"]["source"]
source_b = pair1_structural_pilot["B"]["source"]

a_c1 = candidate_logprob(source_a, c1)
a_c2 = candidate_logprob(source_a, c2)

b_c1 = candidate_logprob(source_b, c1)
b_c2 = candidate_logprob(source_b, c2)

log_odds_a = a_c1 - a_c2
log_odds_b = b_c1 - b_c2

paired_shift = log_odds_a - log_odds_b

print("Source A log-odds (C1 - C2):", log_odds_a)
print("Source B log-odds (C1 - C2):", log_odds_b)
print()
print("Paired structural shift:", paired_shift)
print(
    "Expected-direction shift:",
    paired_shift > 0
)

--- CELL 37 ---
@torch.no_grad()
def source_head_tail_logprob(source, head, tail):
    prefix = (
        source
        + "\n\nThe relation can be summarized as:"
        + head
    )

    prefix_ids = tok.encode(prefix, add_special_tokens=False)
    tail_ids = tok.encode(tail, add_special_tokens=False)

    full_ids = prefix_ids + tail_ids

    input_ids = torch

In [66]:
# 3.38d — M+ on the three comparative-relation pairs
# Uses the frozen cached source/concept difference vectors.
# No model forward passes are rerun.

comparative_pair_ids = [5, 11, 12]

comparative_mplus_results = []

for pair_id in comparative_pair_ids:

    raw_layer_alignments = []
    mplus_layer_alignments = []

    for layer in sweep_layers:

        source_diff = (
            m_diff_cache[pair_id][layer]["source_diff"]
            .float()
            .cpu()
        )

        concept_diff = (
            m_diff_cache[pair_id][layer]["concept_diff"]
            .float()
            .cpu()
        )

        sigma = shared_calibration_stats[layer]["std"]

        # Frozen original M
        raw_alignment = cos(
            source_diff,
            concept_diff
        )

        # M+ = diagonal whitening
        whitened_source_diff = source_diff / sigma
        whitened_concept_diff = concept_diff / sigma

        mplus_alignment = cos(
            whitened_source_diff,
            whitened_concept_diff
        )

        raw_layer_alignments.append(raw_alignment)
        mplus_layer_alignments.append(mplus_alignment)

    raw_arr = np.array(raw_layer_alignments)
    mplus_arr = np.array(mplus_layer_alignments)

    result = {
        "pair_id": pair_id,
        "raw_mean": raw_arr.mean(),
        "mplus_mean": mplus_arr.mean(),
        "raw_positive_layers": int((raw_arr > 0).sum()),
        "mplus_positive_layers": int((mplus_arr > 0).sum()),
        "raw_layers": raw_layer_alignments,
        "mplus_layers": mplus_layer_alignments,
    }

    comparative_mplus_results.append(result)

    print("=" * 65)
    print(f"PAIR {pair_id}")
    print(
        f"Original M mean: {result['raw_mean']:+.4f} "
        f"| positive layers: {result['raw_positive_layers']}/7"
    )
    print(
        f"M+ mean:         {result['mplus_mean']:+.4f} "
        f"| positive layers: {result['mplus_positive_layers']}/7"
    )
    print()
    for layer, raw, mp in zip(
        sweep_layers,
        raw_layer_alignments,
        mplus_layer_alignments
    ):
        print(
            f"L{layer}: "
            f"M={raw:+.4f} | "
            f"M+={mp:+.4f}"
        )

PAIR 5
Original M mean: -0.0757 | positive layers: 0/7
M+ mean:         -0.0471 | positive layers: 2/7

L20: M=-0.1257 | M+=-0.0623
L22: M=-0.1245 | M+=-0.0851
L24: M=-0.0884 | M+=-0.0602
L25: M=-0.0936 | M+=-0.0643
L26: M=-0.0238 | M+=-0.0672
L27: M=-0.0336 | M+=+0.0085
L28: M=-0.0406 | M+=+0.0011
PAIR 11
Original M mean: -0.0128 | positive layers: 2/7
M+ mean:         +0.0240 | positive layers: 3/7

L20: M=-0.1326 | M+=-0.0596
L22: M=-0.0509 | M+=-0.0139
L24: M=-0.0444 | M+=-0.0206
L25: M=-0.0336 | M+=-0.0044
L26: M=-0.0216 | M+=+0.0049
L27: M=+0.0864 | M+=+0.1271
L28: M=+0.1069 | M+=+0.1346
PAIR 12
Original M mean: -0.1472 | positive layers: 0/7
M+ mean:         -0.0941 | positive layers: 0/7

L20: M=-0.1627 | M+=-0.0672
L22: M=-0.1422 | M+=-0.0816
L24: M=-0.1559 | M+=-0.0957
L25: M=-0.1469 | M+=-0.1029
L26: M=-0.1272 | M+=-0.0809
L27: M=-0.1554 | M+=-0.1195
L28: M=-0.1404 | M+=-0.1111


In [67]:
# 3.39 — Permutation null for comparative M+
# Matched statistic: mean M+ alignment across comparative Pairs 5, 11, 12.
# Null: match those three source differences to WRONG held-out concept differences.

import random
import numpy as np

random.seed(42)

comparative_pair_ids = [5, 11, 12]
all_pair_ids = sorted(m_diff_cache.keys())

# --------------------------------------------------
# 1. Matched M+ statistic
# --------------------------------------------------
matched_pair_means = []

for pair_id in comparative_pair_ids:

    layer_vals = []

    for layer in sweep_layers:

        source_diff = (
            m_diff_cache[pair_id][layer]["source_diff"]
            .float()
            .cpu()
        )

        concept_diff = (
            m_diff_cache[pair_id][layer]["concept_diff"]
            .float()
            .cpu()
        )

        sigma = shared_calibration_stats[layer]["std"]

        alignment = cos(
            source_diff / sigma,
            concept_diff / sigma
        )

        layer_vals.append(alignment)

    matched_pair_means.append(
        np.mean(layer_vals)
    )

matched_stat = np.mean(matched_pair_means)


# --------------------------------------------------
# 2. Mismatched null
# --------------------------------------------------
n_permutations = 500
null_stats = []

for _ in range(n_permutations):

    # Pick three distinct wrong concept pairs
    while True:
        wrong_ids = random.sample(
            all_pair_ids,
            len(comparative_pair_ids)
        )

        if all(
            wrong_id != source_id
            for source_id, wrong_id
            in zip(comparative_pair_ids, wrong_ids)
        ):
            break

    perm_pair_means = []

    for source_pair_id, concept_pair_id in zip(
        comparative_pair_ids,
        wrong_ids
    ):

        layer_vals = []

        for layer in sweep_layers:

            source_diff = (
                m_diff_cache[source_pair_id][layer]["source_diff"]
                .float()
                .cpu()
            )

            concept_diff = (
                m_diff_cache[concept_pair_id][layer]["concept_diff"]
                .float()
                .cpu()
            )

            sigma = shared_calibration_stats[layer]["std"]

            alignment = cos(
                source_diff / sigma,
                concept_diff / sigma
            )

            layer_vals.append(alignment)

        perm_pair_means.append(
            np.mean(layer_vals)
        )

    null_stats.append(
        np.mean(perm_pair_means)
    )


null_stats = np.array(null_stats)

# One-sided empirical p:
# how often is the null at least as positive as matched?
empirical_p = (
    1 + np.sum(null_stats >= matched_stat)
) / (
    1 + len(null_stats)
)

print("Comparative matched pair means:")
for pair_id, value in zip(
    comparative_pair_ids,
    matched_pair_means
):
    print(f"  Pair {pair_id}: {value:+.4f}")

print()
print("Matched comparative M+ mean:", matched_stat)
print("Null mean:", null_stats.mean())
print("Null median:", np.median(null_stats))
print("Null 95th percentile:", np.percentile(null_stats, 95))
print("Null max:", null_stats.max())
print("Empirical one-sided p:", empirical_p)

Comparative matched pair means:
  Pair 5: -0.0471
  Pair 11: +0.0240
  Pair 12: -0.0941

Matched comparative M+ mean: -0.03906595714700719
Null mean: -0.004823235557481115
Null median: -0.0028154613793871944
Null 95th percentile: 0.016787270493140174
Null max: 0.03162070864900237
Empirical one-sided p: 1.0


## M+ summary

Whitening moved all three comparative pairs in the positive direction and flipped Pair 11 positive, but did not rescue the category overall.

Comparative mean:

`-0.0786 -> -0.0391`

The whitened comparative result still failed its recomputed permutation null, with empirical one-sided `p = 1.0`.

So whitening softened the failure but did not explain it away.

In [68]:
# 3.41 — Save M+ robustness results

import json
from pathlib import Path

results_dir = Path("/workspace/multi-token-jlens/results")
results_dir.mkdir(exist_ok=True)

mplus_output = {
    "comparative_pair_ids": comparative_pair_ids,
    "pair_results": comparative_mplus_results,
    "matched_stat": float(matched_stat),
    "null_mean": float(null_stats.mean()),
    "null_median": float(np.median(null_stats)),
    "null_95th_percentile": float(np.percentile(null_stats, 95)),
    "null_max": float(null_stats.max()),
    "empirical_one_sided_p": float(empirical_p),
    "n_permutations": int(n_permutations),
    "notes": {
        "status": "post-hoc exploratory robustness check",
        "core_gate3_results_changed": False,
        "effective_change": "diagonal whitening; centering cancels under difference-direction metric"
    }
}

out_path = results_dir / "mplus_comparative_robustness.json"

with open(out_path, "w") as f:
    json.dump(mplus_output, f, indent=2)

print("Saved:")
print(out_path)

Saved:
/workspace/multi-token-jlens/results/mplus_comparative_robustness.json


In [69]:
# 3.42 — Save frozen M permutation verification summary

import random
import numpy as np
import json
from pathlib import Path

random.seed(42)

pair_ids = sorted(m_diff_cache.keys())
n = len(pair_ids)
n_permutations = 500

# Matched statistic
matched_pair_means = []

for pair_id in pair_ids:
    layer_vals = []

    for layer in sweep_layers:
        source_diff = m_diff_cache[pair_id][layer]["source_diff"]
        concept_diff = m_diff_cache[pair_id][layer]["concept_diff"]

        layer_vals.append(
            cos(source_diff, concept_diff)
        )

    matched_pair_means.append(
        np.mean(layer_vals)
    )

matched_stat = np.mean(matched_pair_means)

# Permutation null
null_stats = []

for _ in range(n_permutations):
    while True:
        perm = pair_ids.copy()
        random.shuffle(perm)

        if all(
            perm[i] != pair_ids[i]
            for i in range(n)
        ):
            break

    perm_pair_means = []

    for source_pair_id, concept_pair_id in zip(
        pair_ids,
        perm
    ):
        layer_vals = []

        for layer in sweep_layers:
            source_diff = (
                m_diff_cache[source_pair_id][layer]["source_diff"]
            )
            concept_diff = (
                m_diff_cache[concept_pair_id][layer]["concept_diff"]
            )

            layer_vals.append(
                cos(source_diff, concept_diff)
            )

        perm_pair_means.append(
            np.mean(layer_vals)
        )

    null_stats.append(
        np.mean(perm_pair_means)
    )

null_stats = np.array(null_stats)

empirical_p = (
    1 + np.sum(null_stats >= matched_stat)
) / (
    1 + len(null_stats)
)

summary = {
    "matched_mean": float(matched_stat),
    "null_mean": float(null_stats.mean()),
    "null_median": float(np.median(null_stats)),
    "null_95th_percentile": float(np.percentile(null_stats, 95)),
    "null_max": float(null_stats.max()),
    "empirical_one_sided_p": float(empirical_p),
    "n_permutations": int(n_permutations),
}

out_path = Path(
    "/workspace/multi-token-jlens/results/"
    "m_permutation_verification.json"
)

with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(summary)
print()
print("Saved:")
print(out_path)

{'matched_mean': 0.04200427130230571, 'null_mean': -0.0020694622421870007, 'null_median': -0.002591174505104558, 'null_95th_percentile': 0.017852476586549686, 'null_max': 0.03935324800039204, 'empirical_one_sided_p': 0.001996007984031936, 'n_permutations': 500}

Saved:
/workspace/multi-token-jlens/results/m_permutation_verification.json


In [70]:
# 3.43 — Verify positive-pair-count permutation p-value

import random
import numpy as np
import json
from pathlib import Path

random.seed(42)

pair_ids = sorted(m_diff_cache.keys())

matched_pair_means = np.array([
    np.mean([
        cos(
            m_diff_cache[pair_id][layer]["source_diff"],
            m_diff_cache[pair_id][layer]["concept_diff"]
        )
        for layer in sweep_layers
    ])
    for pair_id in pair_ids
])

matched_positive_pairs = int(
    (matched_pair_means > 0).sum()
)

n_permutations = 500
null_positive_counts = []

for _ in range(n_permutations):

    while True:
        shuffled_ids = pair_ids.copy()
        random.shuffle(shuffled_ids)

        if all(
            shuffled_ids[i] != pair_ids[i]
            for i in range(len(pair_ids))
        ):
            break

    perm_pair_means = []

    for source_pair, concept_pair in zip(
        pair_ids,
        shuffled_ids
    ):

        layer_vals = []

        for layer in sweep_layers:

            layer_vals.append(
                cos(
                    m_diff_cache[source_pair][layer]["source_diff"],
                    m_diff_cache[concept_pair][layer]["concept_diff"]
                )
            )

        perm_pair_means.append(
            np.mean(layer_vals)
        )

    null_positive_counts.append(
        int(
            (
                np.array(perm_pair_means) > 0
            ).sum()
        )
    )

null_positive_counts = np.array(
    null_positive_counts
)

p_positive = (
    1
    + np.sum(
        null_positive_counts
        >= matched_positive_pairs
    )
) / (
    n_permutations + 1
)

summary = {
    "matched_positive_pairs": matched_positive_pairs,
    "null_mean_positive_count": float(
        null_positive_counts.mean()
    ),
    "null_95th_percentile_positive_count": float(
        np.percentile(
            null_positive_counts,
            95
        )
    ),
    "empirical_one_sided_p": float(
        p_positive
    ),
    "n_permutations": n_permutations,
}

print(summary)

out_path = Path(
    "/workspace/multi-token-jlens/results/"
    "m_positive_count_verification.json"
)

with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print()
print("Saved:")
print(out_path)

{'matched_positive_pairs': 9, 'null_mean_positive_count': 6.576, 'null_95th_percentile_positive_count': 9.049999999999955, 'empirical_one_sided_p': 0.1497005988023952, 'n_permutations': 500}

Saved:
/workspace/multi-token-jlens/results/m_positive_count_verification.json


In [71]:
# 3.44 — Check cached L1 sanity-check objects

names_to_check = [
    "matched_shifts",
    "mismatched_shifts",
    "permutation_mean_abs_shifts",
    "heldout_l1_results",
    "l1_mismatch_results",
]

for name in names_to_check:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name}: FOUND | "
            f"type={type(obj).__name__} | "
            f"len={len(obj) if hasattr(obj, '__len__') else 'n/a'}"
        )
    else:
        print(f"{name}: NOT FOUND")

matched_shifts: FOUND | type=ndarray | len=14
mismatched_shifts: FOUND | type=ndarray | len=14
permutation_mean_abs_shifts: FOUND | type=list | len=20
heldout_l1_results: FOUND | type=list | len=14
l1_mismatch_results: FOUND | type=list | len=14


In [72]:
# 3.45 — Verify L1 matched/null magnitude ratio

import numpy as np
import json
from pathlib import Path

matched_mean_abs = np.abs(matched_shifts).mean()
mismatched_mean_abs = np.abs(mismatched_shifts).mean()

null_mean_abs = np.mean(permutation_mean_abs_shifts)
null_median_abs = np.median(permutation_mean_abs_shifts)
null_max_abs = np.max(permutation_mean_abs_shifts)

matched_vs_mismatched_ratio = (
    matched_mean_abs / mismatched_mean_abs
)

matched_vs_null_ratio = (
    matched_mean_abs / null_mean_abs
)

summary = {
    "matched_mean_abs_shift": float(matched_mean_abs),
    "mismatched_mean_abs_shift": float(mismatched_mean_abs),
    "matched_vs_mismatched_ratio": float(matched_vs_mismatched_ratio),
    "permutation_null_mean_abs_shift": float(null_mean_abs),
    "permutation_null_median_abs_shift": float(null_median_abs),
    "permutation_null_max_abs_shift": float(null_max_abs),
    "matched_vs_permutation_null_ratio": float(matched_vs_null_ratio),
    "n_permutations": len(permutation_mean_abs_shifts),
}

for k, v in summary.items():
    print(f"{k}: {v}")

out_path = Path(
    "/workspace/multi-token-jlens/results/"
    "l1_magnitude_verification.json"
)

with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print()
print("Saved:")
print(out_path)

matched_mean_abs_shift: 2.1645989554268974
mismatched_mean_abs_shift: 0.09307105654761896
matched_vs_mismatched_ratio: 23.257487727340884
permutation_null_mean_abs_shift: 0.1370492408389137
permutation_null_median_abs_shift: 0.11686023530505953
permutation_null_max_abs_shift: 0.23643043154761895
matched_vs_permutation_null_ratio: 15.794315548023686
n_permutations: 20

Saved:
/workspace/multi-token-jlens/results/l1_magnitude_verification.json
